In [ ]:
# same step 4 repeat can be skipped, as the output is already generated in step 4
import json, csv, os
import pandas as pd

# Load Step 3 selected OOV words
with open("oov_selected_words.json", encoding="utf-8") as f:
    step3 = json.load(f)

df_oov = pd.DataFrame(step3["words"])
print(f"OOV words loaded : {len(df_oov):,}")
print(df_oov.head(10).to_string(index=False))


OOV words loaded : 2,000
       word  corpus_freq missing_bigrams  n_missing
      झडङ्ङ            5        [डङ, ्ङ]          2
    डङ्ङगुर            4        [डङ, ्ङ]          2
       डङ्ङ            3        [डङ, ्ङ]          2
      तङ्ङु            2        [ङु, ्ङ]          2
     ढुङ्ङो            2        [ङो, ्ङ]          2
  एफइआइएमएस            7        [इआ, फइ]          2
  एफइआईएमएस            2        [इआ, फइ]          2
एर्फइआईएमएस            2        [इआ, फइ]          2
  भृ्ङ्गराज            2        [ृ्, ्ङ]          2
   हृवाङ्ङै            3        [ङै, ्ङ]          2


In [ ]:
import re

# Heuristic pre-classifier
# Assigns a suggested category to each word.
# A Nepali language expert should review and correct these.

# Patterns
LATIN_RE    = re.compile(r'[A-Za-z]')                         # contains any Latin char
ALL_CAPS_RE = re.compile(r'^[A-Z]{2,}$')                      # pure uppercase acronym
DEVA_RE     = re.compile(r'^[\u0900-\u097F\s]+$')             # pure Devanagari

# Known place-name suffixes in Nepali
PLACE_SUFFIXES = ('गर', 'पुर', 'गञ्ज', 'नगर', 'बजार', 'चोक', 'टोल',
                  'खोला', 'दह', 'ताल', 'डाँडा', 'गाउँ', 'थान', 'कुण्ड')

# Known person-name suffixes / titles
PERSON_SUFFIXES = ('लाल', 'बहादुर', 'प्रसाद', 'कुमार', 'देवी', 'माया',
                   'राज', 'मान', 'धर', 'दास')

# Known govt / scheme keywords
GOVT_KEYWORDS = ('योजना', 'कार्यक्रम', 'आयोग', 'समिति', 'बोर्ड',
                 'मन्त्रालय', 'विभाग', 'कोष', 'निगम', 'प्राधिकरण')

# Known company keywords
CMPY_KEYWORDS = ('बैंक', 'इन्स्योरेन्स', 'फाइनान्स', 'ग्रुप',
                 'लिमिटेड', 'कर्पोरेशन', 'इन्टरनेशनल', 'नेपाल')

def suggest_category(word: str) -> str:
    # 1. Pure Latin acronym
    if ALL_CAPS_RE.match(word):
        return 'ABBR'
    # 2. Mixed Latin+Devanagari → code-mixed
    if LATIN_RE.search(word) and DEVA_RE.search(word):
        return 'CM'
    # 3. Pure Latin (brand / acronym)
    if LATIN_RE.search(word):
        return 'BRAND'
    # 4. Govt scheme keywords
    if any(kw in word for kw in GOVT_KEYWORDS):
        return 'GOVT'
    # 5. Company keywords
    if any(kw in word for kw in CMPY_KEYWORDS):
        return 'CMPY'
    # 6. Place-name suffixes
    if any(word.endswith(s) for s in PLACE_SUFFIXES):
        return 'PROP'
    # 7. Person-name suffixes
    if any(word.endswith(s) for s in PERSON_SUFFIXES):
        return 'PROP'
    # 8. Navigation keywords
    if any(kw in word for kw in ('चोक', 'मार्ग', 'सडक', 'रोड', 'पथ')):
        return 'NAV'
    # 9. Default — proper noun (most OOV Devanagari words are names)
    return 'PROP'

df_oov['category_auto'] = df_oov['word'].apply(suggest_category)
df_oov['category']      = df_oov['category_auto']   # expert will override this column
df_oov['is_oov']        = True
df_oov['notes']         = ''
df_oov['keep']          = True   # expert sets False to drop noise/toxic words

print("Auto-classification distribution:")
print(df_oov['category_auto'].value_counts().to_string())


Auto-classification distribution:
category_auto
PROP    1993
NAV        4
GOVT       2
CMPY       1


In [ ]:
# Flag likely noise for expert review

PURE_DEVA = re.compile(r'^[\u0900-\u0963\u0972-\u097F]+$')

def flag_noise(word: str) -> str:
    flags = []
    # Repeated consonant cluster (onomatopoeia / OCR artifact)
    if re.search(r'(.)\1{2,}', word):
        flags.append('repeated_chars')
    # Very short
    if len(word) <= 2:
        flags.append('too_short')
    # Very long
    if len(word) >= 20:
        flags.append('too_long')
    # Contains digits
    if re.search(r'\d', word):
        flags.append('has_digits')
    # Halant-nga cluster (्ङ) — often OCR noise
    if '्ङ' in word and word.count('ङ') >= 2:
        flags.append('double_nga_cluster')
    return '|'.join(flags) if flags else ''

df_oov['noise_flags'] = df_oov['word'].apply(flag_noise)

noisy = df_oov[df_oov['noise_flags'] != '']
print(f"Words flagged for review : {len(noisy):,} / {len(df_oov):,}")
print(f"\nFlag breakdown:")
from collections import Counter
all_flags = [f for flags in noisy['noise_flags'] for f in flags.split('|')]
for flag, cnt in Counter(all_flags).most_common():
    print(f"  {flag:25s} : {cnt}")
print(f"\nSample flagged words:")
print(noisy[['word','corpus_freq','category','noise_flags']].head(20).to_string(index=False))


Words flagged for review : 90 / 2,000

Flag breakdown:
  too_short                 : 72
  double_nga_cluster        : 9
  repeated_chars            : 6
  too_long                  : 3

Sample flagged words:
                 word  corpus_freq category        noise_flags
                झडङ्ङ            5     PROP double_nga_cluster
              डङ्ङगुर            4     PROP double_nga_cluster
                 डङ्ङ            3     PROP double_nga_cluster
                तङ्ङु            2     PROP double_nga_cluster
               ढुङ्ङो            2     PROP double_nga_cluster
             हृवाङ्ङै            3     PROP double_nga_cluster
             ड्याङ्ङै            2     PROP double_nga_cluster
                 छङ्ङ            2     PROP double_nga_cluster
            ठङ्ग्रङ्ङ            2     PROP double_nga_cluster
                   इख           91     PROP          too_short
                   नई           99     PROP          too_short
                   यफ           37   

In [7]:
os.makedirs("step4_results", exist_ok=True)

# Full review sheet — expert opens this, corrects 'category' and 'keep' columns
review_cols = ['word', 'corpus_freq', 'n_missing', 'missing_bigrams',
               'category', 'is_oov', 'keep', 'noise_flags', 'notes']

df_review = df_oov[review_cols].copy()
# Sort: clean words first, noisy at bottom
df_review = df_review.sort_values(['noise_flags', 'category', 'corpus_freq'],
                                   ascending=[True, True, False])

review_path = "step4_results/oov_words_for_review.csv"
df_review.to_csv(review_path, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel

print(f"✓ Saved → {review_path}")
print(f"\nCategory distribution (pre-expert review):")
print(df_review['category'].value_counts().to_string())
print(f"\nWords flagged keep=True  : {df_review['keep'].sum():,}")
print(f"Words flagged keep=False : {(~df_review['keep']).sum():,}")
print(f"\n→ Share oov_words_for_review.csv with a Nepali language expert.")
print(f"  Expert should:")
print(f"  1. Set 'keep=False' for noise, toxic, or misspelled words")
print(f"  2. Correct 'category' column (ABBR/BRAND/CM/CMPY/GOVT/PROP/NAV)")
print(f"  3. Fill 'notes' for anything unusual")


✓ Saved → step4_results/oov_words_for_review.csv

Category distribution (pre-expert review):
category
PROP    1993
NAV        4
GOVT       2
CMPY       1

Words flagged keep=True  : 2,000
Words flagged keep=False : 0

→ Share oov_words_for_review.csv with a Nepali language expert.
  Expert should:
  1. Set 'keep=False' for noise, toxic, or misspelled words
  2. Correct 'category' column (ABBR/BRAND/CM/CMPY/GOVT/PROP/NAV)
  3. Fill 'notes' for anything unusual


In [ ]:
# Run this AFTER the expert returns the reviewed CSV
# Loads the corrected file, filters noise, and saves one CSV per category
# matching the benchmark file structure from the guide.

reviewed_path = "step4_results/oov_words_for_review.csv"  # replace with expert-reviewed file

df_reviewed = pd.read_csv(reviewed_path, encoding='utf-8-sig')

# Keep only approved words
df_clean = df_reviewed[df_reviewed['keep'] == True].copy()
print(f"Words after expert filter : {len(df_clean):,} / {len(df_reviewed):,}")

# Per-category breakdown
print(f"\nCategory breakdown:")
print(df_clean['category'].value_counts().to_string())

# Save one CSV per category
CATEGORIES = ['ABBR', 'BRAND', 'CM', 'CMPY', 'GOVT', 'PROP', 'NAV']
os.makedirs("step4_results/by_category", exist_ok=True)

for cat in CATEGORIES:
    cat_df = df_clean[df_clean['category'] == cat][['word', 'category', 'is_oov', 'notes']]
    out = f"step4_results/by_category/{cat}.csv"
    cat_df.to_csv(out, index=False, encoding='utf-8-sig')
    print(f"  {cat:6s} → {len(cat_df):4d} words  saved → {out}")

print(f"\n→ Categories with < 50 words need supplementing (Step 4 guide: target 50 OOV per category).")
short = [(cat, len(df_clean[df_clean['category']==cat])) for cat in CATEGORIES
          if len(df_clean[df_clean['category']==cat]) < 50]
if short:
    for cat, n in short:
        print(f"  {cat}: only {n} words — expert should add manually from seed list")
else:
    print("  All categories have ≥ 50 words ✓")


Words after expert filter : 2,000 / 2,000

Category breakdown:
category
PROP    1993
NAV        4
GOVT       2
CMPY       1
  ABBR   →    0 words  saved → step4_results/by_category/ABBR.csv
  BRAND  →    0 words  saved → step4_results/by_category/BRAND.csv
  CM     →    0 words  saved → step4_results/by_category/CM.csv
  CMPY   →    1 words  saved → step4_results/by_category/CMPY.csv
  GOVT   →    2 words  saved → step4_results/by_category/GOVT.csv
  PROP   → 1993 words  saved → step4_results/by_category/PROP.csv
  NAV    →    4 words  saved → step4_results/by_category/NAV.csv

→ Categories with < 50 words need supplementing (Step 4 guide: target 50 OOV per category).
  ABBR: only 0 words — expert should add manually from seed list
  BRAND: only 0 words — expert should add manually from seed list
  CM: only 0 words — expert should add manually from seed list
  CMPY: only 1 words — expert should add manually from seed list
  GOVT: only 2 words — expert should add manually from seed list

# Step 4 — Supplement Thin Categories + Collect IV Control Words

**Goal:** Every category needs ≥ 50 OOV + 50 IV words before Step 5.
This notebook:
1. Loads the Step 4 per-category CSVs
2. Injects seed OOV words from the benchmark guide for thin categories (target: **75 OOV per category** — 50% buffer)
3. Samples **75 IV words per category** from the TTS training vocab as control group
4. Merges everything and saves final `step4_results/by_category/<CAT>.csv`
5. Prints a readiness report


In [12]:
import json, os, re, random
import pandas as pd
from pathlib import Path

random.seed(42)

CATEGORIES = ['ABBR', 'BRAND', 'CM', 'CMPY', 'GOVT', 'PROP', 'NAV']
TARGET_OOV = 75
TARGET_IV  = 75

step4_dir = Path('step4_results/by_category')
cat_dfs = {}
for cat in CATEGORIES:
    p = step4_dir / f'{cat}.csv'
    if p.exists():
        df = pd.read_csv(p, encoding='utf-8-sig')
        cat_dfs[cat] = df
        print(f'  {cat:6s} loaded : {len(df):4d} OOV words')
    else:
        cat_dfs[cat] = pd.DataFrame(columns=['word','category','is_oov','notes'])
        print(f'  {cat:6s} : file not found — starting empty')


  ABBR   loaded :    0 OOV words
  BRAND  loaded :    0 OOV words
  CM     loaded :    0 OOV words
  CMPY   loaded :    1 OOV words
  GOVT   loaded :    2 OOV words
  PROP   loaded : 1993 OOV words
  NAV    loaded :    4 OOV words


In [ ]:
# Seed OOV words extracted from the benchmark guide appendix.
# Each entry: (word, romanization_or_expansion, notes)
# Romanization stored in the 'notes' column for the expert's reference.


SEED_OOV = {

'ABBR': [
    # From guide + extended set — all pure Latin acronyms
    ('ICIMOD', 'ICIMOD', 'Intl Centre for Integrated Mountain Development'),
    ('SAARC', 'SAARC', 'South Asian Association for Regional Cooperation'),
    ('UNESCO', 'UNESCO', 'UN Educational Scientific Cultural Organization'),
    ('UNICEF', 'UNICEF', 'UN Childrens Fund'),
    ('UNDP', 'UNDP', 'UN Development Programme'),
    ('NRB', 'NRB', 'Nepal Rastra Bank'),
    ('FNCCI', 'FNCCI', 'Federation of Nepalese Chambers of Commerce'),
    ('CIAA', 'CIAA', 'Commission for Investigation of Abuse of Authority'),
    ('OPM', 'OPM', 'Office of Prime Minister'),
    ('BIPPA', 'BIPPA', 'Bilateral Investment Promotion and Protection Agreement'),
    ('SEBON', 'SEBON', 'Securities Board of Nepal'),
    ('BNMT', 'BNMT', 'Border Network and Mobile Telecom'),
    ('UCPN', 'UCPN', 'Unified Communist Party of Nepal'),
    ('NHRC', 'NHRC', 'National Human Rights Commission'),
    ('WHO', 'WHO', 'World Health Organization'),
    ('IMF', 'IMF', 'International Monetary Fund'),
    ('ADB', 'ADB', 'Asian Development Bank'),
    ('WFP', 'WFP', 'World Food Programme'),
    ('UNHCR', 'UNHCR', 'UN High Commissioner for Refugees'),
    ('NATO', 'NATO', 'North Atlantic Treaty Organization'),
    ('ASEAN', 'ASEAN', 'Association of Southeast Asian Nations'),
    ('ICC', 'ICC', 'International Criminal Court'),
    ('INGO', 'INGO', 'International Non-Governmental Organization'),
    ('NGO', 'NGO', 'Non-Governmental Organization'),
    ('MOU', 'MOU', 'Memorandum of Understanding'),
    ('GDP', 'GDP', 'Gross Domestic Product'),
    ('GNP', 'GNP', 'Gross National Product'),
    ('FDI', 'FDI', 'Foreign Direct Investment'),
    ('VAT', 'VAT', 'Value Added Tax'),
    ('ATM', 'ATM', 'Automated Teller Machine'),
    ('PIN', 'PIN', 'Personal Identification Number'),
    ('OTP', 'OTP', 'One Time Password'),
    ('SIM', 'SIM', 'Subscriber Identity Module'),
    ('GPS', 'GPS', 'Global Positioning System'),
    ('USB', 'USB', 'Universal Serial Bus'),
    ('WiFi', 'WiFi', 'Wireless Fidelity — mixed case'),
    ('CCTV', 'CCTV', 'Closed Circuit Television'),
    ('PCR', 'PCR', 'Polymerase Chain Reaction — COVID context'),
    ('ICU', 'ICU', 'Intensive Care Unit'),
    ('ECG', 'ECG', 'Electrocardiogram'),
    ('MRI', 'MRI', 'Magnetic Resonance Imaging'),
    ('CT', 'CT', 'Computed Tomography scan'),
    ('BC', 'BC', 'Before Christ — calendar'),
    ('AD', 'AD', 'Anno Domini'),
    ('VS', 'VS', 'Versus'),
    ('PDF', 'PDF', 'Portable Document Format'),
    ('SMS', 'SMS', 'Short Message Service'),
    ('TV', 'TV', 'Television'),
    ('FM', 'FM', 'Frequency Modulation'),
    ('AM', 'AM', 'Amplitude Modulation / morning'),
    ('PM', 'PM', 'Prime Minister / afternoon'),
    ('KTM', 'KTM', 'Kathmandu — airport code'),
    ('PKR', 'PKR', 'Pakistani Rupee'),
    ('INR', 'INR', 'Indian Rupee'),
    ('USD', 'USD', 'US Dollar'),
    ('EUR', 'EUR', 'Euro'),
    ('GBP', 'GBP', 'British Pound Sterling'),
    ('NPR', 'NPR', 'Nepali Rupee — currency code'),
    ('HR', 'HR', 'Human Resources'),
    ('IT', 'IT', 'Information Technology'),
    ('AI', 'AI', 'Artificial Intelligence'),
    ('ML', 'ML', 'Machine Learning'),
    ('API', 'API', 'Application Programming Interface'),
    ('URL', 'URL', 'Uniform Resource Locator'),
    ('IP', 'IP', 'Internet Protocol'),
    ('OS', 'OS', 'Operating System'),
    ('RAM', 'RAM', 'Random Access Memory'),
    ('ROM', 'ROM', 'Read Only Memory'),
    ('CPU', 'CPU', 'Central Processing Unit'),
    ('GPU', 'GPU', 'Graphics Processing Unit'),
    ('SSD', 'SSD', 'Solid State Drive'),
    ('HDD', 'HDD', 'Hard Disk Drive'),
    ('VPN', 'VPN', 'Virtual Private Network'),
    ('IoT', 'IoT', 'Internet of Things'),
    ('EV', 'EV', 'Electric Vehicle'),
    ('ICIMOD',   'ICIMOD',   'Intl Centre for Integrated Mountain Development'),
    ('SAARC',    'SAARC',    'South Asian Association for Regional Cooperation'),
    ('UNESCO',   'UNESCO',   'UN Educational Scientific Cultural Organization'),
    ('UNICEF',   'UNICEF',   'UN Childrens Fund'),
    ('UNDP',     'UNDP',     'UN Development Programme'),
    ('NRB',      'NRB',      'Nepal Rastra Bank'),
    ('FNCCI',    'FNCCI',    'Federation of Nepalese Chambers of Commerce'),
    ('CIAA',     'CIAA',     'Commission for Investigation of Abuse of Authority'),
    ('OPM',      'OPM',      'Office of Prime Minister'),
    ('BIPPA',    'BIPPA',    'Bilateral Investment Promotion and Protection Agreement'),
    ('SEBON',    'SEBON',    'Securities Board of Nepal'),
    ('BNMT',     'BNMT',     'Border Network and Mobile Telecom'),
    ('UCPN',     'UCPN',     'Unified Communist Party of Nepal'),
    ('NHRC',     'NHRC',     'National Human Rights Commission'),
    ('WHO',      'WHO',      'World Health Organization'),
    ('IMF',      'IMF',      'International Monetary Fund'),
    ('ADB',      'ADB',      'Asian Development Bank'),
    ('WFP',      'WFP',      'World Food Programme'),
    ('UNHCR',    'UNHCR',    'UN High Commissioner for Refugees'),
    ('NATO',     'NATO',     'North Atlantic Treaty Organization'),
    ('ASEAN',    'ASEAN',    'Association of Southeast Asian Nations'),
    ('ICC',      'ICC',      'International Criminal Court'),
    ('INGO',     'INGO',     'International Non-Governmental Organization'),
    ('NGO',      'NGO',      'Non-Governmental Organization'),
    ('MOU',      'MOU',      'Memorandum of Understanding'),
    ('GDP',      'GDP',      'Gross Domestic Product'),
    ('GNP',      'GNP',      'Gross National Product'),
    ('FDI',      'FDI',      'Foreign Direct Investment'),
    ('VAT',      'VAT',      'Value Added Tax'),
    ('ATM',      'ATM',      'Automated Teller Machine'),
    ('PIN',      'PIN',      'Personal Identification Number'),
    ('OTP',      'OTP',      'One Time Password'),
    ('SIM',      'SIM',      'Subscriber Identity Module'),
    ('GPS',      'GPS',      'Global Positioning System'),
    ('USB',      'USB',      'Universal Serial Bus'),
    ('WiFi',     'WiFi',     'Wireless Fidelity — mixed case'),
    ('CCTV',     'CCTV',     'Closed Circuit Television'),
    ('PCR',      'PCR',      'Polymerase Chain Reaction — COVID context'),
    ('ICU',      'ICU',      'Intensive Care Unit'),
    ('ECG',      'ECG',      'Electrocardiogram'),
    ('MRI',      'MRI',      'Magnetic Resonance Imaging'),
    ('CT',       'CT',       'Computed Tomography scan'),
    ('BC',       'BC',       'Before Christ — calendar'),
    ('AD',       'AD',       'Anno Domini'),
    ('VS',       'VS',       'Versus'),
    ('PDF',      'PDF',      'Portable Document Format'),
    ('SMS',      'SMS',      'Short Message Service'),
    ('TV',       'TV',       'Television'),
    ('FM',       'FM',       'Frequency Modulation'),
    ('AM',       'AM',       'Amplitude Modulation / morning'),
    ('PM',       'PM',       'Prime Minister / afternoon'),
    ('KTM',      'KTM',      'Kathmandu — airport code'),
    ('PKR',      'PKR',      'Pakistani Rupee'),
    ('INR',      'INR',      'Indian Rupee'),
    ('USD',      'USD',      'US Dollar'),
    ('EUR',      'EUR',      'Euro'),
    ('GBP',      'GBP',      'British Pound Sterling'),
    ('NPR',      'NPR',      'Nepali Rupee — currency code'),
    ('HR',       'HR',       'Human Resources'),
    ('IT',       'IT',       'Information Technology'),
    ('AI',       'AI',       'Artificial Intelligence'),
    ('ML',       'ML',       'Machine Learning'),
    ('API',      'API',      'Application Programming Interface'),
    ('URL',      'URL',      'Uniform Resource Locator'),
    ('IP',       'IP',       'Internet Protocol'),
    ('OS',       'OS',       'Operating System'),
    ('RAM',      'RAM',      'Random Access Memory'),
    ('ROM',      'ROM',      'Read Only Memory'),
    ('CPU',      'CPU',      'Central Processing Unit'),
    ('GPU',      'GPU',      'Graphics Processing Unit'),
    ('SSD',      'SSD',      'Solid State Drive'),
    ('HDD',      'HDD',      'Hard Disk Drive'),
    ('VPN',      'VPN',      'Virtual Private Network'),
    ('IoT',      'IoT',      'Internet of Things'),
    ('EV',       'EV',       'Electric Vehicle'),
    ('NGO',      'NGO',      'Non-governmental organisation (duplicate marker)'),
],

'BRAND': [
    # Local + international brands present in Nepali media
    ('Goldstar', 'Goldstar', 'CG Electronics brand — final consonant cluster'),
    ('Wai Wai', 'Wai Wai', 'Instant noodle brand — reduplication'),
    ('Dabur', 'Dabur', 'FMCG brand — final -ur'),
    ('Colgate', 'Colgate', 'Dental care — English loan'),
    ('Lux', 'Lux', 'Soap brand — consonant cluster final'),
    ('Sprite', 'Sprite', 'Beverage — Spr- cluster'),
    ('Fanta', 'Fanta', 'Beverage — Fn cluster'),
    ('Kurkure', 'Kurkure', 'Snack — retroflex'),
    ('Maggi', 'Maggi', 'Noodle brand — geminate G'),
    ('Khalti', 'Khalti', 'Digital wallet — retroflex'),
    ('Bhatbhateni', 'Bhatbhateni', 'Supermarket chain — aspirated + retroflex'),
    ('Daraz', 'Daraz', 'E-commerce — Arabic-origin final z'),
    ('Samsung', 'Samsung', 'Electronics — ng-final'),
    ('Nokia', 'Nokia', 'Mobile — oi diphthong'),
    ('Xiaomi', 'Xiaomi', 'Mobile — Xia- cluster unusual in Nepali'),
    ('Oppo', 'Oppo', 'Mobile — geminate p'),
    ('Vivo', 'Vivo', 'Mobile — v-initial'),
    ('Realme', 'Realme', 'Mobile — eal cluster'),
    ('OnePlus', 'OnePlus', 'Mobile — compound + plus'),
    ('Huawei', 'Huawei', 'Chinese brand — Hw- cluster'),
    ('Pepsi', 'Pepsi', 'Beverage — ps cluster'),
    ('Coca-Cola', 'Coca-Cola', 'Beverage — hyphenated'),
    ('Lipton', 'Lipton', 'Tea brand — consonant cluster'),
    ('Nescafe', 'Nescafe', 'Coffee — sc- cluster'),
    ('Oreo', 'Oreo', 'Biscuit — vowel sequence'),
    ('Lays', 'Lays', 'Chips — ay diphthong'),
    ('Surf Excel', 'Surf Excel', 'Detergent — compound'),
    ('Ariel', 'Ariel', 'Detergent — iel cluster'),
    ('Dettol', 'Dettol', 'Antiseptic — geminate t + final l'),
    ('Panadol', 'Panadol', 'Medicine — final -ol'),
    ('Disprin', 'Disprin', 'Medicine — spr- cluster'),
    ('Zandu', 'Zandu', 'Ayurvedic brand — z-initial'),
    ('Himalaya', 'Himalaya', 'Herbal brand — aya suffix'),
    ('Patanjali', 'Patanjali', 'Ayurvedic — retroflex cluster'),
    ('Honda', 'Honda', 'Vehicle — nd cluster'),
    ('Suzuki', 'Suzuki', 'Vehicle — z + ki'),
    ('Yamaha', 'Yamaha', 'Vehicle/music — aha'),
    ('Bajaj', 'Bajaj', 'Vehicle — final j'),
    ('Tata', 'Tata', 'Conglomerate — simple but brand'),
    ('Hyundai', 'Hyundai', 'Vehicle — Hy- + diphthong ai'),
    ('Maruti', 'Maruti', 'Vehicle — u-final'),
    ('Bira', 'Bira', 'Beer brand — v/b alternation'),
    ('Tuborg', 'Tuborg', 'Beer — final rg cluster'),
    ('Carlsberg', 'Carlsberg', 'Beer — final rg cluster'),
    ('Snickers', 'Snickers', 'Candy — Sn- + kers'),
    ('KitKat', 'KitKat', 'Chocolate — compound'),
    ('Cadbury', 'Cadbury', 'Chocolate — dbury cluster'),
    ('Puma', 'Puma', 'Sportswear — final -a'),
    ('Adidas', 'Adidas', 'Sportswear — das final'),
    ('Nike', 'Nike', 'Sportswear — silent e in English'),
    ('Levi', 'Levi', 'Jeans — i-final'),
    ('Reebok', 'Reebok', 'Sportswear — ee + bok'),
    ('Bata', 'Bata', 'Footwear — simple but brand'),
    ('CG', 'CG', 'Chaudhary Group — abbreviation brand'),
    ('IME', 'IME', 'Remittance brand — acronym'),
    ('Esewa', 'Esewa', 'Digital payment — e-initial'),
    ('FonePay', 'FonePay', 'Payment app — compound'),
    ('ConnectIPS', 'ConnectIPS', 'Payment — camelCase'),
    ('Sasto Deal', 'Sasto Deal', 'E-commerce — code-mix'),
    ('Hamro Bazar', 'Hamro Bazar', 'E-commerce — Nepali+bazar'),
    ('OldDurbar', 'OldDurbar', 'Whisky brand — compound'),
    ('Ruslan', 'Ruslan', 'Vodka brand — final n'),
    ('Surf', 'Surf', 'Detergent — common short brand'),
    ('Closeup', 'Closeup', 'Toothpaste — compound'),
    ('Lifebuoy', 'Lifebuoy', 'Soap — buoy cluster'),
    ('Dove', 'Dove', 'Personal care — v-final'),
    ('Pantene', 'Pantene', 'Haircare — nt- cluster'),
    ('Head Shoulders', 'Head Shoulders', 'Shampoo — compound'),
    ('Gillette', 'Gillette', 'Razor — double l + ette'),
    ('Sensodyne', 'Sensodyne', 'Toothpaste — dyne suffix'),
    ('Bosch', 'Bosch', 'Electronics — sch final'),
    ('LG', 'LG', 'Electronics — two-letter brand'),
    ('Panasonic', 'Panasonic', 'Electronics — onic suffix'),
    ('Philips', 'Philips', 'Electronics — Ph- + lps'),
    ('Whirlpool', 'Whirlpool', 'Appliance — Wh- + pool'),
     ('Goldstar',      'Goldstar',      'CG Electronics brand — final consonant cluster'),
    ('Wai Wai',       'Wai Wai',       'Instant noodle brand — reduplication'),
    ('Dabur',         'Dabur',         'FMCG brand — final -ur'),
    ('Colgate',       'Colgate',       'Dental care — English loan'),
    ('Lux',           'Lux',           'Soap brand — consonant cluster final'),
    ('Sprite',        'Sprite',        'Beverage — Spr- cluster'),
    ('Fanta',         'Fanta',         'Beverage — Fn cluster'),
    ('Kurkure',       'Kurkure',       'Snack — retroflex'),
    ('Maggi',         'Maggi',         'Noodle brand — geminate G'),
    ('Khalti',        'Khalti',        'Digital wallet — retroflex'),
    ('Bhatbhateni',   'Bhatbhateni',   'Supermarket chain — aspirated + retroflex'),
    ('Daraz',         'Daraz',         'E-commerce — Arabic-origin final z'),
    ('Samsung',       'Samsung',       'Electronics — ng-final'),
    ('Nokia',         'Nokia',         'Mobile — oi diphthong'),
    ('Xiaomi',        'Xiaomi',        'Mobile — Xia- cluster unusual in Nepali'),
    ('Oppo',          'Oppo',          'Mobile — geminate p'),
    ('Vivo',          'Vivo',          'Mobile — v-initial'),
    ('Realme',        'Realme',        'Mobile — eal cluster'),
    ('OnePlus',       'OnePlus',       'Mobile — compound + plus'),
    ('Huawei',        'Huawei',        'Chinese brand — Hw- cluster'),
    ('Pepsi',         'Pepsi',         'Beverage — ps cluster'),
    ('Coca-Cola',     'Coca-Cola',     'Beverage — hyphenated'),
    ('Lipton',        'Lipton',        'Tea brand — consonant cluster'),
    ('Nescafe',       'Nescafe',       'Coffee — sc- cluster'),
    ('Oreo',          'Oreo',          'Biscuit — vowel sequence'),
    ('Lays',          'Lays',          'Chips — ay diphthong'),
    ('Surf Excel',    'Surf Excel',    'Detergent — compound'),
    ('Ariel',         'Ariel',         'Detergent — iel cluster'),
    ('Dettol',        'Dettol',        'Antiseptic — geminate t + final l'),
    ('Panadol',       'Panadol',       'Medicine — final -ol'),
    ('Disprin',       'Disprin',       'Medicine — spr- cluster'),
    ('Zandu',         'Zandu',         'Ayurvedic brand — z-initial'),
    ('Himalaya',      'Himalaya',      'Herbal brand — aya suffix'),
    ('Patanjali',     'Patanjali',     'Ayurvedic — retroflex cluster'),
    ('Honda',         'Honda',         'Vehicle — nd cluster'),
    ('Suzuki',        'Suzuki',        'Vehicle — z + ki'),
    ('Yamaha',        'Yamaha',        'Vehicle/music — aha'),
    ('Bajaj',         'Bajaj',         'Vehicle — final j'),
    ('Tata',          'Tata',          'Conglomerate — simple but brand'),
    ('Hyundai',       'Hyundai',       'Vehicle — Hy- + diphthong ai'),
    ('Maruti',        'Maruti',        'Vehicle — u-final'),
    ('Bira',          'Bira',          'Beer brand — v/b alternation'),
    ('Tuborg',        'Tuborg',        'Beer — final rg cluster'),
    ('Carlsberg',     'Carlsberg',     'Beer — final rg cluster'),
    ('Snickers',      'Snickers',      'Candy — Sn- + kers'),
    ('KitKat',        'KitKat',        'Chocolate — compound'),
    ('Cadbury',       'Cadbury',       'Chocolate — dbury cluster'),
    ('Puma',          'Puma',          'Sportswear — final -a'),
    ('Adidas',        'Adidas',        'Sportswear — das final'),
    ('Nike',          'Nike',          'Sportswear — silent e in English'),
    ('Levi',          'Levi',          'Jeans — i-final'),
    ('Reebok',        'Reebok',        'Sportswear — ee + bok'),
    ('Bata',          'Bata',          'Footwear — simple but brand'),
    ('CG',            'CG',            'Chaudhary Group — abbreviation brand'),
    ('IME',           'IME',           'Remittance brand — acronym'),
    ('Esewa',         'Esewa',         'Digital payment — e-initial'),
    ('FonePay',       'FonePay',       'Payment app — compound'),
    ('ConnectIPS',    'ConnectIPS',    'Payment — camelCase'),
    ('Sasto Deal',    'Sasto Deal',    'E-commerce — code-mix'),
    ('Hamro Bazar',   'Hamro Bazar',   'E-commerce — Nepali+bazar'),
    ('OldDurbar',     'OldDurbar',     'Whisky brand — compound'),
    ('Ruslan',        'Ruslan',        'Vodka brand — final n'),
],

'CM': [
    # Code-mixed phrases: Nepali+English as seen in urban Nepali text
    ('online बुकिङ', 'online buking', 'E-booking — English noun + Nepali suffix'),
    ('digital payment', 'digital payment', 'All-English phrase in Nepali context'),
    ('smartphone चार्ज', 'smartphone charge', 'Sph- cluster + Nepali'),
    ('freelancing काम', 'freelancing kaam', 'English gerund + Nepali noun'),
    ('internet bandwidth', 'internet bandwidth', 'Technical English in Nepali'),
    ('EMI मा किन्नुस्', 'EMI ma kinnusu', 'Abbreviation in Nepali syntax'),
    ('OTP पठाउनुस्', 'OTP pathaunusu', 'Acronym in Nepali sentence'),
    ('CCTV जडान', 'CCTV jadan', 'Acronym + Nepali noun'),
    ('YouTube channel', 'YouTube channel', 'Compound English in Nepali'),
    ('password बिर्सनु', 'password birsanu', 'English noun + Nepali verb'),
    ('screenshot लिनुस्', 'screenshot linusu', 'English noun + Nepali imperative'),
    ('WhatsApp group', 'WhatsApp group', 'App name + English noun'),
    ('photoshop गर्नुस्', 'photoshop garnusu', 'Brand-as-verb + Nepali'),
    ('electricity bill online', 'electricity bill online', 'Full English phrase'),
    ('mobile recharge', 'mobile recharge', 'English compound in Nepali media'),
    ('fast internet', 'fast internet', 'English adj+noun'),
    ('cloud storage', 'cloud storage', 'Technical English'),
    ('video call गर्ने', 'video call garne', 'English compound + Nepali infinitive'),
    ('selfie खिच्नुस्', 'selfie khichnusu', 'English loanword + Nepali verb'),
    ('trending topic', 'trending topic', 'Social media English'),
    ('live streaming', 'live streaming', 'Media English'),
    ('data plan', 'data plan', 'Telecom English'),
    ('roaming charge', 'roaming charge', 'Telecom English'),
    ('SIM swap', 'SIM swap', 'Telecom — acronym + English'),
    ('cyber crime', 'cyber crime', 'Legal English in Nepali'),
    ('phishing attack', 'phishing attack', 'Cybersecurity English'),
    ('spam message', 'spam message', 'Email/tech English'),
    ('app update', 'app update', 'Short English compound'),
    ('software install', 'software install', 'Technical English'),
    ('hardware problem', 'hardware problem', 'Technical English'),
    ('laptop charge', 'laptop charge', 'Device English'),
    ('WiFi signal', 'WiFi signal', 'Technical English'),
    ('Bluetooth connect', 'Bluetooth connect', 'Technical English'),
    ('GPS location', 'GPS location', 'Navigation English'),
    ('delivery tracking', 'delivery tracking', 'E-commerce English'),
    ('cashback offer', 'cashback offer', 'Finance English'),
    ('discount code', 'discount code', 'Retail English'),
    ('voucher apply', 'voucher apply', 'Retail English'),
    ('refund request', 'refund request', 'Consumer English'),
    ('account login', 'account login', 'Tech English'),
    ('profile update', 'profile update', 'Social media English'),
    ('news feed', 'news feed', 'Social media English'),
    ('story post', 'story post', 'Social media English'),
    ('like comment', 'like comment', 'Social media English'),
    ('share गर्नुस्', 'share garnusu', 'English verb + Nepali'),
    ('download गर्नुस्', 'download garnusu', 'English verb + Nepali'),
    ('upload भयो', 'upload bhayo', 'English verb + Nepali past'),
    ('login गर्नुस्', 'login garnusu', 'English noun-verb + Nepali'),
    ('reboot गर्नुस्', 'reboot garnusu', 'Technical English + Nepali'),
    ('format गर्नुस्', 'format garnusu', 'Technical English + Nepali'),
    ('backup लिनुस्', 'backup linusu', 'Technical English + Nepali'),
    ('reset गर्नुस्', 'reset garnusu', 'Technical English + Nepali'),
    ('upgrade गर्नुस्', 'upgrade garnusu', 'Technical English + Nepali'),
    ('cancel गर्नुस्', 'cancel garnusu', 'English + Nepali'),
    ('confirm गर्नुस्', 'confirm garnusu', 'English + Nepali'),
    ('submit गर्नुस्', 'submit garnusu', 'English + Nepali'),
    ('block गर्नुस्', 'block garnusu', 'Social media English + Nepali'),
    ('report गर्नुस्', 'report garnusu', 'Social media English + Nepali'),
    ('unfollow गर्नुस्', 'unfollow garnusu', 'Social media English + Nepali'),
    ('mute गर्नुस्', 'mute garnusu', 'Social media English + Nepali'),
    ('online class', 'online class', 'Education English'),
    ('zoom meeting', 'zoom meeting', 'Video call platform'),
    ('Google search', 'Google search', 'Search engine compound'),
    ('Facebook post', 'Facebook post', 'Social media compound'),
    ('Instagram reel', 'Instagram reel', 'Social media English'),
    ('TikTok video', 'TikTok video', 'Platform + content type'),
    ('email पठाउनुस्', 'email pathaunusu', 'English + Nepali imperative'),
    ('print गर्नुस्', 'print garnusu', 'Technical English + Nepali'),
    ('scan गर्नुस्', 'scan garnusu', 'Technical English + Nepali'),
    ('copy गर्नुस्', 'copy garnusu', 'Technical English + Nepali'),
    ('paste गर्नुस्', 'paste garnusu', 'Technical English + Nepali'),
    ('delete गर्नुस्', 'delete garnusu', 'Technical English + Nepali'),
    ('subscribe गर्नुस्', 'subscribe garnusu', 'Social media English + Nepali'),
    ('notification आयो', 'notification aayo', 'English noun + Nepali past'),
    ('typing गर्दैछ', 'typing gardaichha', 'English gerund + Nepali continuous'),
     ('online बुकिङ',          'online buking',      'E-booking — English noun + Nepali suffix'),
    ('digital payment',       'digital payment',    'All-English phrase in Nepali context'),
    ('smartphone चार्ज',      'smartphone charge',  'Sph- cluster + Nepali'),
    ('freelancing काम',       'freelancing kaam',   'English gerund + Nepali noun'),
    ('internet bandwidth',    'internet bandwidth', 'Technical English in Nepali'),
    ('EMI मा किन्नुस्',       'EMI ma kinnusu',     'Abbreviation in Nepali syntax'),
    ('OTP पठाउनुस्',          'OTP pathaunusu',     'Acronym in Nepali sentence'),
    ('CCTV जडान',             'CCTV jadan',         'Acronym + Nepali noun'),
    ('YouTube channel',       'YouTube channel',    'Compound English in Nepali'),
    ('password बिर्सनु',      'password birsanu',   'English noun + Nepali verb'),
    ('screenshot लिनुस्',     'screenshot linusu',  'English noun + Nepali imperative'),
    ('WhatsApp group',        'WhatsApp group',     'App name + English noun'),
    ('photoshop गर्नुस्',     'photoshop garnusu',  'Brand-as-verb + Nepali'),
    ('electricity bill online','electricity bill online','Full English phrase'),
    ('mobile recharge',       'mobile recharge',    'English compound in Nepali media'),
    ('fast internet',         'fast internet',      'English adj+noun'),
    ('cloud storage',         'cloud storage',      'Technical English'),
    ('video call गर्ने',      'video call garne',   'English compound + Nepali infinitive'),
    ('selfie खिच्नुस्',       'selfie khichnusu',   'English loanword + Nepali verb'),
    ('trending topic',        'trending topic',     'Social media English'),
    ('live streaming',        'live streaming',     'Media English'),
    ('data plan',             'data plan',          'Telecom English'),
    ('roaming charge',        'roaming charge',     'Telecom English'),
    ('SIM swap',              'SIM swap',           'Telecom — acronym + English'),
    ('cyber crime',           'cyber crime',        'Legal English in Nepali'),
    ('phishing attack',       'phishing attack',    'Cybersecurity English'),
    ('spam message',          'spam message',       'Email/tech English'),
    ('app update',            'app update',         'Short English compound'),
    ('software install',      'software install',   'Technical English'),
    ('hardware problem',      'hardware problem',   'Technical English'),
    ('laptop charge',         'laptop charge',      'Device English'),
    ('WiFi signal',           'WiFi signal',        'Technical English'),
    ('Bluetooth connect',     'Bluetooth connect',  'Technical English'),
    ('GPS location',          'GPS location',       'Navigation English'),
    ('delivery tracking',     'delivery tracking',  'E-commerce English'),
    ('cashback offer',        'cashback offer',     'Finance English'),
    ('discount code',         'discount code',      'Retail English'),
    ('voucher apply',         'voucher apply',      'Retail English'),
    ('refund request',        'refund request',     'Consumer English'),
    ('account login',         'account login',      'Tech English'),
    ('profile update',        'profile update',     'Social media English'),
    ('news feed',             'news feed',          'Social media English'),
    ('story post',            'story post',         'Social media English'),
    ('like comment',          'like comment',       'Social media English'),
    ('share गर्नुस्',         'share garnusu',      'English verb + Nepali'),
    ('download गर्नुस्',      'download garnusu',   'English verb + Nepali'),
    ('upload भयो',            'upload bhayo',       'English verb + Nepali past'),
    ('login गर्नुस्',         'login garnusu',      'English noun-verb + Nepali'),
    ('reboot गर्नुस्',        'reboot garnusu',     'Technical English + Nepali'),
    ('format गर्नुस्',        'format garnusu',     'Technical English + Nepali'),
    ('backup लिनुस्',         'backup linusu',      'Technical English + Nepali'),
    ('reset गर्नुस्',         'reset garnusu',      'Technical English + Nepali'),
    ('upgrade गर्नुस्',       'upgrade garnusu',    'Technical English + Nepali'),
    ('cancel गर्नुस्',        'cancel garnusu',     'English + Nepali'),
    ('confirm गर्नुस्',       'confirm garnusu',    'English + Nepali'),
    ('submit गर्नुस्',        'submit garnusu',     'English + Nepali'),
    ('block गर्नुस्',         'block garnusu',      'Social media English + Nepali'),
    ('report गर्नुस्',        'report garnusu',     'Social media English + Nepali'),
    ('unfollow गर्नुस्',      'unfollow garnusu',   'Social media English + Nepali'),
    ('mute गर्नुस्',          'mute garnusu',       'Social media English + Nepali'),
],

'CMPY': [
    ('Ncell', 'Ncell', 'Telecom — N-initial + consonant cluster'),
    ('Worldlink', 'Worldlink', 'ISP — Wor- cluster'),
    ('Vianet', 'Vianet', 'ISP — Via-'),
    ('Subisu', 'Subisu', 'ISP — unusual vowel sequence'),
    ('Nepal Telecom', 'Nepal Telecom', 'State telecom company'),
    ('Prabhu Money', 'Prabhu Money', 'Remittance — aspirated + English'),
    ('IME Remit', 'IME Remit', 'Remittance — acronym + English'),
    ('Muktinath Bikas Bank', 'Muktinath Bikas Bank', 'Bank — conjunct cluster'),
    ('Himalayan Bank', 'Himalayan Bank', 'Bank — English adjective'),
    ('Machhapuchchhre Bank', 'Machhapuchchhre Bank', 'Bank — double conjunct'),
    ('Rastriya Banijya Bank', 'Rastriya Banijya Bank', 'State bank — conjunct'),
    ('Standard Chartered', 'Standard Chartered', 'Bank — English compound'),
    ('Everest Bank', 'Everest Bank', 'Bank — English name'),
    ('Nepal Investment Bank', 'Nepal Investment Bank', 'Bank — English compound'),
    ('NIC Asia Bank', 'NIC Asia Bank', 'Bank — acronym + region'),
    ('Laxmi Sunrise Bank', 'Laxmi Sunrise Bank', 'Bank — compound English'),
    ('Citizens Bank', 'Citizens Bank', 'Bank — English'),
    ('Sanima Bank', 'Sanima Bank', 'Bank — Nepali+English'),
    ('Nabil Bank', 'Nabil Bank', 'Bank — Arabic-origin name'),
    ('Kumari Bank', 'Kumari Bank', 'Bank — Nepali goddess'),
    ('Sunrise Bank', 'Sunrise Bank', 'Bank — English compound'),
    ('Prime Commercial Bank', 'Prime Commercial Bank', 'Bank — English'),
    ('Nepal SBI Bank', 'Nepal SBI Bank', 'Bank — acronym'),
    ('Prabhu Bank', 'Prabhu Bank', 'Bank — Nepali+English'),
    ('Siddhartha Bank', 'Siddhartha Bank', 'Bank — Sanskrit name'),
    ('Global IME Bank', 'Global IME Bank', 'Bank — acronym compound'),
    ('Agriculture Development Bank', 'Agriculture Development Bank', 'State bank — full compound'),
    ('Shikhar Insurance', 'Shikhar Insurance', 'Insurance — Nepali+English'),
    ('Nepal Life Insurance', 'Nepal Life Insurance', 'Insurance — compound'),
    ('Sagarmatha Insurance', 'Sagarmatha Insurance', 'Insurance — mountain name'),
    ('Neco Insurance', 'Neco Insurance', 'Insurance — acronym'),
    ('Bouddha Air', 'Bouddha Air', 'Airline — religious name'),
    ('Yeti Airlines', 'Yeti Airlines', 'Airline — Nepali myth'),
    ('Buddha Air', 'Buddha Air', 'Airline — Sanskrit'),
    ('Saurya Airlines', 'Saurya Airlines', 'Airline — Sanskrit'),
    ('Tara Air', 'Tara Air', 'Airline — Sanskrit'),
    ('Shree Airlines', 'Shree Airlines', 'Airline — Sanskrit title'),
    ('CG Corp Global', 'CG Corp Global', 'Conglomerate — acronym'),
    ('Chaudhary Group', 'Chaudhary Group', 'Conglomerate — aspirated'),
    ('Golchha Organisation', 'Golchha Organisation', 'Conglomerate — lchh cluster'),
    ('Soaltee Hotel', 'Soaltee Hotel', 'Hospitality — unusual vowel'),
    ('Hyatt Regency', 'Hyatt Regency', 'Hotel — Hy- + ency'),
    ('Marriott', 'Marriott', 'Hotel — double r + iott'),
    ('Himalayan Java', 'Himalayan Java', 'Cafe chain — Java = tech/coffee'),
    ('Roadhouse Cafe', 'Roadhouse Cafe', 'Restaurant chain — compound'),
    ('Bhatbhateni Superstore', 'Bhatbhateni Superstore', 'Retail — aspirated + English'),
    ('Saleways', 'Saleways', 'Retail — ways suffix'),
    ('BigMart', 'BigMart', 'Retail — compound'),
    ('Sastodeal', 'Sastodeal', 'E-commerce — Nepali+deal'),
    ('Muncha House', 'Muncha House', 'Restaurant — Newari+English'),
    ('Karnali Hydropower', 'Karnali Hydropower', 'Energy — compound'),
    ('Chilime Hydropower', 'Chilime Hydropower', 'Energy — place name'),
    ('Nepal Oil Corporation', 'Nepal Oil Corporation', 'SOE — full compound'),
    ('Hetauda Cement', 'Hetauda Cement', 'SOE — place + English'),
    ('Dish Home', 'Dish Home', 'DTH service — compound'),
    ('iSmart', 'iSmart', 'ISP — i-prefix camelCase'),
    ('Classic Tech', 'Classic Tech', 'ISP — English compound'),
    ('Websurfer', 'Websurfer', 'ISP — compound'),
    ('Nepal Airlines', 'Nepal Airlines', 'National carrier'),
    ('Summit Air', 'Summit Air', 'Airline — English compound'),
    ('Simrik Airlines', 'Simrik Airlines', 'Airline — Simrik place name'),
    ('Muktinath Development Bank', 'Muktinath Development Bank', 'Bank — long compound'),
    ('Mega Bank', 'Mega Bank', 'Bank — Greek prefix'),
    ('Civil Bank', 'Civil Bank', 'Bank — English'),
    ('Century Commercial Bank', 'Century Commercial Bank', 'Bank — century prefix'),
    ('Janata Bank', 'Janata Bank', 'Bank — Nepali+English'),
    ('Purnima Finance', 'Purnima Finance', 'Finance — Sanskrit+English'),
    ('NIBL Ace Capital', 'NIBL Ace Capital', 'Finance — acronym + English'),
    ('Nabil Invest', 'Nabil Invest', 'Securities — Arabic+English'),
    ('Sewa Laghubitta', 'Sewa Laghubitta', 'Microfinance — Nepali'),
    ('Deprosc Laghubitta', 'Deprosc Laghubitta', 'Microfinance — acronym'),
    ('NMB Bank', 'NMB Bank', 'Bank — acronym'),
    ('Kamana Sewa Bank', 'Kamana Sewa Bank', 'Development bank — Nepali'),
    ('Garima Bikas Bank', 'Garima Bikas Bank', 'Development bank — Nepali'),
    ('Ncell',                  'Ncell',               'Telecom — N-initial + consonant cluster'),
    ('Worldlink',              'Worldlink',            'ISP — Wor- cluster'),
    ('Vianet',                 'Vianet',               'ISP — Via-'),
    ('Subisu',                 'Subisu',               'ISP — unusual vowel sequence'),
    ('Nepal Telecom',          'Nepal Telecom',        'State telecom company'),
    ('Prabhu Money',           'Prabhu Money',         'Remittance — aspirated + English'),
    ('IME Remit',              'IME Remit',            'Remittance — acronym + English'),
    ('Muktinath Bikas Bank',   'Muktinath Bikas Bank', 'Bank — conjunct cluster'),
    ('Himalayan Bank',         'Himalayan Bank',       'Bank — English adjective'),
    ('Machhapuchchhre Bank',   'Machhapuchchhre Bank', 'Bank — double conjunct'),
    ('Rastriya Banijya Bank',  'Rastriya Banijya Bank','State bank — conjunct'),
    ('Standard Chartered',     'Standard Chartered',   'Bank — English compound'),
    ('Everest Bank',           'Everest Bank',         'Bank — English name'),
    ('Nepal Investment Bank',  'Nepal Investment Bank','Bank — English compound'),
    ('NIC Asia Bank',          'NIC Asia Bank',        'Bank — acronym + region'),
    ('Laxmi Sunrise Bank',     'Laxmi Sunrise Bank',   'Bank — compound English'),
    ('Citizens Bank',          'Citizens Bank',        'Bank — English'),
    ('Sanima Bank',            'Sanima Bank',          'Bank — Nepali+English'),
    ('Nabil Bank',             'Nabil Bank',           'Bank — Arabic-origin name'),
    ('Kumari Bank',            'Kumari Bank',          'Bank — Nepali goddess'),
    ('Sunrise Bank',           'Sunrise Bank',         'Bank — English compound'),
    ('Prime Commercial Bank',  'Prime Commercial Bank','Bank — English'),
    ('Nepal SBI Bank',         'Nepal SBI Bank',       'Bank — acronym'),
    ('Prabhu Bank',            'Prabhu Bank',          'Bank — Nepali+English'),
    ('Siddhartha Bank',        'Siddhartha Bank',      'Bank — Sanskrit name'),
    ('Global IME Bank',        'Global IME Bank',      'Bank — acronym compound'),
    ('Agriculture Development Bank','ADB Nepal',       'State bank — full compound'),
    ('Shikhar Insurance',      'Shikhar Insurance',    'Insurance — Nepali+English'),
    ('Nepal Life Insurance',   'Nepal Life Insurance', 'Insurance — compound'),
    ('Sagarmatha Insurance',   'Sagarmatha Insurance', 'Insurance — mountain name'),
    ('Neco Insurance',         'Neco Insurance',       'Insurance — acronym'),
    ('Bouddha Air',            'Bouddha Air',          'Airline — religious name'),
    ('Yeti Airlines',          'Yeti Airlines',        'Airline — Nepali myth'),
    ('Buddha Air',             'Buddha Air',           'Airline — Sanskrit'),
    ('Saurya Airlines',        'Saurya Airlines',      'Airline — Sanskrit'),
    ('Tara Air',               'Tara Air',             'Airline — Sanskrit'),
    ('Shree Airlines',         'Shree Airlines',       'Airline — Sanskrit title'),
    ('CG Corp Global',         'CG Corp Global',       'Conglomerate — acronym'),
    ('Chaudhary Group',        'Chaudhary Group',      'Conglomerate — aspirated'),
    ('Golchha Organisation',   'Golchha Organisation', 'Conglomerate — lchh cluster'),
    ('Soaltee Hotel',          'Soaltee Hotel',        'Hospitality — unusual vowel'),
    ('Hyatt Regency',          'Hyatt Regency',        'Hotel — Hy- + ency'),
    ('Marriott',               'Marriott',             'Hotel — double r + iott'),
    ('Himalayan Java',         'Himalayan Java',       'Cafe chain — Java = tech/coffee'),
    ('Roadhouse Cafe',         'Roadhouse Cafe',       'Restaurant chain — compound'),
    ('Bhatbhateni Superstore', 'Bhatbhateni Superstore','Retail — aspirated + English'),
    ('Saleways',               'Saleways',             'Retail — ways suffix'),
    ('BigMart',                'BigMart',              'Retail — compound'),
    ('Sastodeal',              'Sastodeal',            'E-commerce — Nepali+deal'),
    ('Muncha House',           'Muncha House',         'Restaurant — Newari+English'),
    ('Karnali Hydropower',     'Karnali Hydropower',   'Energy — compound'),
    ('Chilime Hydropower',     'Chilime Hydropower',   'Energy — place name'),
    ('Nepal Oil Corporation',  'Nepal Oil Corporation','SOE — full compound'),
    ('Hetauda Cement',         'Hetauda Cement',       'SOE — place + English'),
],

'GOVT': [
    # Government programs, ministries, schemes — Nepali names
    ('उज्ज्वला योजना', 'Ujjwala Yojana', 'Jj- geminate conjunct'),
    ('आत्मनिर्भर', 'Aatmanirbhar', 'Sanskrit compound — self-reliant'),
    ('प्रधानमन्त्री आवास', 'Pradhanmantri Awas', 'Long compound — PM housing scheme'),
    ('समृद्ध नेपाल', 'Samridhha Nepal', 'Conjunct ddh'),
    ('दिगो विकास', 'Digo Bikas', 'Sustainable development'),
    ('राष्ट्रिय स्वास्थ्य बिमा', 'Rastriya Swasthya Bima', 'Conjunct sw- + retroflex'),
    ('सुकुम्बासी आवास', 'Sukumbasi Awas', 'Landless housing scheme'),
    ('कृषि ऋण', 'Krishi Rin', 'Agricultural loan scheme'),
    ('बालिका विवाह उत्थान', 'Balika Bihe Utthan', 'Girl child marriage uplift scheme'),
    ('कर्णाली एकीकृत', 'Karnali Integrated', 'Code-mix scheme name'),
    ('मुख्यमन्त्री रोजगार', 'Mukhyamantri Rojgar', 'CM employment scheme — conjunct'),
    ('युवा स्वरोजगार', 'Yuwa Swarojgar', 'Youth self-employment — sw-'),
    ('अनुदान कार्यक्रम', 'Anudan Karyakram', 'Subsidy program'),
    ('सामाजिक सुरक्षा', 'Samajik Suraksha', 'Social security — cluster'),
    ('वृद्ध भत्ता', 'Bridha Bhatta', 'Old age allowance'),
    ('विधवा भत्ता', 'Bidhawa Bhatta', 'Widow allowance — dh- cluster'),
    ('अपाङ्गता भत्ता', 'Apangata Bhatta', 'Disability allowance — conjunct ङ्ग'),
    ('एकल महिला', 'Ekal Mahila', 'Single women scheme'),
    ('जनजाति उत्थान', 'Janajati Utthan', 'Indigenous uplift — double t'),
    ('दलित सशक्तिकरण', 'Dalit Sashaktikaran', 'Dalit empowerment — sh+kt cluster'),
    ('सिंचाइ विकास', 'Sinchai Bikas', 'Irrigation development'),
    ('खानेपानी योजना', 'Khanepani Yojana', 'Drinking water scheme'),
    ('सडक विस्तार', 'Sadak Bistar', 'Road expansion — retroflex'),
    ('विद्युत् प्राधिकरण', 'Bidhyut Pradhikaran', 'Electricity authority — conjunct'),
    ('नेपाल विद्युत्', 'Nepal Bidhyut', 'NEA — conjunct t'),
    ('राजस्व अनुसन्धान', 'Rajaswa Anusandhan', 'Revenue investigation — sw-'),
    ('भ्रष्टाचार नियन्त्रण', 'Bhrashtachar Niyantran', 'Corruption control — conjunct'),
    ('संविधान संशोधन', 'Sambidhan Sanshodhan', 'Constitution amendment — conjunct'),
    ('निर्वाचन आयोग', 'Nirwachan Aayog', 'Election commission'),
    ('लोकसेवा आयोग', 'Loksewa Aayog', 'Public service commission — sw-'),
    ('प्रहरी प्रशिक्षण', 'Prahari Prashikshan', 'Police training — sh+k cluster'),
    ('सैनिक प्रशिक्षण', 'Sainik Prashikshan', 'Military training'),
    ('न्यायपालिका', 'Nyayapalika', 'Judiciary — ny- cluster'),
    ('व्यवस्थापिका', 'Wyawasthapika', 'Legislature — wy- cluster'),
    ('कार्यपालिका', 'Karyapalika', 'Executive — ry- cluster'),
    ('स्थानीय सरकार', 'Sthaniya Sarkar', 'Local government — sth- cluster'),
    ('प्रदेश सरकार', 'Pradesh Sarkar', 'Provincial government'),
    ('संघीय संसद', 'Sanghiya Sansad', 'Federal parliament — conjunct'),
    ('राष्ट्रिय सभा', 'Rastriya Sabha', 'National assembly — conjunct'),
    ('प्रतिनिधि सभा', 'Pratinidhi Sabha', 'House of representatives'),
    ('मन्त्रिपरिषद्', 'Mantriparishad', 'Council of ministers — conjunct'),
    ('अख्तियार दुरुपयोग', 'Akhtiyar Durupayog', 'CIAA — Persian-origin word'),
    ('महालेखा परीक्षक', 'Mahalekha Parikshak', 'Auditor general — conjunct'),
    ('सर्वोच्च अदालत', 'Sarwochcha Adalat', 'Supreme court — conjunct chch'),
    ('उच्च अदालत', 'Uchchha Adalat', 'High court — chh aspirate'),
    ('जिल्ला अदालत', 'Jilla Adalat', 'District court — ll cluster'),
    ('भूमि व्यवस्था', 'Bhumi Wyawastha', 'Land management — wy- cluster'),
    ('वन विभाग', 'Ban Bibhag', 'Forest department'),
    ('पर्यटन बोर्ड', 'Paryatan Board', 'Tourism board — code-mix'),
    ('लगानी बोर्ड', 'Lagani Board', 'Investment board — code-mix'),
    ('गृह मन्त्रालय', 'Griha Mantralaya', 'Home ministry — conjunct'),
    ('परराष्ट्र मन्त्रालय', 'Pararastra Mantralaya', 'Foreign affairs — conjunct'),
    ('अर्थ मन्त्रालय', 'Artha Mantralaya', 'Finance ministry'),
    ('शिक्षा मन्त्रालय', 'Shiksha Mantralaya', 'Education ministry — conjunct'),
    ('स्वास्थ्य मन्त्रालय', 'Swasthya Mantralaya', 'Health ministry — sw- cluster'),
    ('कृषि मन्त्रालय', 'Krishi Mantralaya', 'Agriculture ministry — kr- cluster'),
    ('ऊर्जा मन्त्रालय', 'Urja Mantralaya', 'Energy ministry — vowel initial'),
    ('भौतिक पूर्वाधार', 'Bhautik Purwaadhar', 'Physical infrastructure — conjunct'),
    ('महिला बालबालिका', 'Mahila Balbaalika', 'Women and children ministry'),
    ('युवा तथा खेलकुद', 'Yuwa Khel Kud', 'Youth and sports — kh- cluster'),
    ('सञ्चार तथा सूचना', 'Sanchar Suchana', 'Communications ministry — conjunct'),
    ('वातावरण मन्त्रालय', 'Watawaran Mantralaya', 'Environment ministry'),
    ('पर्यटन मन्त्रालय', 'Paryatan Mantralaya', 'Tourism ministry — ry- cluster'),
    ('उद्योग वाणिज्य', 'Udhyog Wanijya', 'Industry and commerce — conjunct'),
    ('श्रम मन्त्रालय', 'Shram Mantralaya', 'Labour ministry — shr- cluster'),
    ('भूमि व्यवस्थापन', 'Bhumi Byawasthaapan', 'Land management ministry — wy-'),
    ('जलस्रोत विभाग', 'Jalasrot Bibhag', 'Water resources dept — conjunct'),
    ('नागरिक उड्डयन', 'Nagarik Uddayan', 'Civil aviation — dd- conjunct'),
    ('खाद्य व्यवस्था', 'Khadya Byawastha', 'Food management — kh- cluster'),
    ('राष्ट्रिय योजना आयोग', 'Rastriya Yojana Aayog', 'National planning commission'),
    ('केन्द्रीय अनुसन्धान', 'Kendriya Anusandhan', 'Central investigation — conjunct'),
    ('विशेष अदालत', 'Bishesh Adalat', 'Special court'),
    ('महान्यायाधिवक्ता', 'Mahanayayaadhiwakta', 'Attorney general — conjunct'),
    ('उज्ज्वला योजना',        'Ujjwala Yojana',      'Jj- geminate conjunct'),
    ('आत्मनिर्भर',            'Aatmanirbhar',        'Sanskrit compound — self-reliant'),
    ('प्रधानमन्त्री आवास',    'Pradhanmantri Awas',  'Long compound — PM housing scheme'),
    ('समृद्ध नेपाल',          'Samridhha Nepal',     'Conjunct ddh'),
    ('दिगो विकास',            'Digo Bikas',          'Sustainable development'),
    ('राष्ट्रिय स्वास्थ्य बिमा','Rastriya Swasthya Bima','Conjunct sw- + retroflex'),
    ('सुकुम्बासी आवास',       'Sukumbasi Awas',      'Landless housing scheme'),
    ('कृषि ऋण',               'Krishi Rin',          'Agricultural loan scheme'),
    ('बालिका विवाह उत्थान',   'Balika Bihe Utthan',  'Girl child marriage uplift scheme'),
    ('कर्णाली एकीकृत',        'Karnali Integrated',  'Code-mix scheme name'),
    ('मुख्यमन्त्री रोजगार',   'Mukhyamantri Rojgar', 'CM employment scheme — conjunct'),
    ('युवा स्वरोजगार',        'Yuwa Swarojgar',      'Youth self-employment — sw-'),
    ('अनुदान कार्यक्रम',      'Anudan Karyakram',    'Subsidy program'),
    ('सामाजिक सुरक्षा',       'Samajik Suraksha',    'Social security — cluster'),
    ('वृद्ध भत्ता',           'Bridha Bhatta',       'Old age allowance'),
    ('विधवा भत्ता',           'Bidhawa Bhatta',      'Widow allowance — dh- cluster'),
    ('अपाङ्गता भत्ता',        'Apangata Bhatta',     'Disability allowance — conjunct ङ्ग'),
    ('एकल महिला',             'Ekal Mahila',         'Single women scheme'),
    ('जनजाति उत्थान',         'Janajati Utthan',     'Indigenous uplift — double t'),
    ('दलित सशक्तिकरण',       'Dalit Sashaktikaran', 'Dalit empowerment — sh+kt cluster'),
    ('सिंचाइ विकास',          'Sinchai Bikas',       'Irrigation development'),
    ('खानेपानी योजना',        'Khanepani Yojana',    'Drinking water scheme'),
    ('सडक विस्तार',           'Sadak Bistar',        'Road expansion — retroflex'),
    ('विद्युत् प्राधिकरण',    'Bidhyut Pradhikaran', 'Electricity authority — conjunct'),
    ('नेपाल विद्युत्',        'Nepal Bidhyut',       'NEA — conjunct t'),
    ('राजस्व अनुसन्धान',      'Rajaswa Anusandhan',  'Revenue investigation — sw-'),
    ('भ्रष्टाचार नियन्त्रण',  'Bhrashtachar Niyantran','Corruption control — conjunct'),
    ('संविधान संशोधन',        'Sambidhan Sanshodhan','Constitution amendment — conjunct'),
    ('निर्वाचन आयोग',         'Nirwachan Aayog',     'Election commission'),
    ('लोकसेवा आयोग',          'Loksewa Aayog',       'Public service commission — sw-'),
    ('प्रहरी प्रशिक्षण',      'Prahari Prashikshan', 'Police training — sh+k cluster'),
    ('सैनिक प्रशिक्षण',       'Sainik Prashikshan',  'Military training'),
    ('न्यायपालिका',            'Nyayapalika',         'Judiciary — ny- cluster'),
    ('व्यवस्थापिका',          'Wyawasthapika',       'Legislature — wy- cluster'),
    ('कार्यपालिका',           'Karyapalika',         'Executive — ry- cluster'),
    ('स्थानीय सरकार',         'Sthaniya Sarkar',     'Local government — sth- cluster'),
    ('प्रदेश सरकार',          'Pradesh Sarkar',       'Provincial government'),
    ('संघीय संसद',            'Sanghiya Sansad',     'Federal parliament — conjunct'),
    ('राष्ट्रिय सभा',         'Rastriya Sabha',      'National assembly — conjunct'),
    ('प्रतिनिधि सभा',         'Pratinidhi Sabha',    'House of representatives'),
    ('मन्त्रिपरिषद्',         'Mantriparishad',      'Council of ministers — conjunct'),
    ('अख्तियार दुरुपयोग',     'Akhtiyar Durupayog',  'CIAA — Persian-origin word'),
    ('महालेखा परीक्षक',       'Mahalekha Parikshak', 'Auditor general — conjunct'),
    ('सर्वोच्च अदालत',        'Sarwochcha Adalat',   'Supreme court — conjunct chch'),
    ('उच्च अदालत',            'Uchchha Adalat',      'High court — chh aspirate'),
    ('जिल्ला अदालत',          'Jilla Adalat',        'District court — ll cluster'),
    ('भूमि व्यवस्था',         'Bhumi Wyawastha',     'Land management — wy- cluster'),
    ('वन विभाग',              'Ban Bibhag',           'Forest department'),
    ('पर्यटन बोर्ड',          'Paryatan Board',      'Tourism board — code-mix'),
    ('लगानी बोर्ड',           'Lagani Board',        'Investment board — code-mix'),
],

'PROP': [
    # Supplement PROP with rarer place/person names with unusual clusters
    # (Step 4 already has 1993 — only add truly rare bigram words)
    ('ल्हासा', 'Lhasa', 'Lh- cluster — Tibetan'),
    ('ल्होत्से', 'Lhotse', 'Lh- cluster — mountain'),
    ('मकालु', 'Makalu', 'Borrowed Tibetan/Sanskrit'),
    ('मुस्ताङ', 'Mustang', 'Ng-final — region name'),
    ('खोटाङ', 'Khotang', 'Kh- + nasal final'),
    ('सोलुखुम्बु', 'Solukhumbu', 'Kh- cluster + labial'),
    ('सिन्धुपाल्चोक', 'Sindhupalchok', 'Conjunct ल्च'),
    ('दोलखा', 'Dolkha', 'Lkh cluster'),
    ('मनाङ', 'Manang', 'Ng-final'),
    ('डोल्पा', 'Dolpa', 'Retroflex D + final a'),
    ('ह्युम्बु', 'Hyumbu', 'Hy- + Tibetan origin'),
    ('त्सोरोलपा', 'Tso Rolpa', 'Tso- glacier lake — Tibetan'),
    ('फ्वाङलुङ', 'Phwanglung', 'Phw- cluster — Tibetan place'),
    ('ग्ल्यासिअर', 'Glacier', 'English loanword as place reference'),
    ('ज्वालामुखी', 'Jwalamukhi', 'Jw- cluster — volcanic'),
    ('श्रीस्वस्थानी', 'Shree Swasthani', 'Shree + sw- cluster'),
    ('फ्रान्स', 'France', 'Fr- cluster — country name'),
    ('ब्राजिल', 'Brazil', 'Br- + final l'),
    ('जर्मनी', 'Germany', 'Jrm- cluster'),
    ('क्यानडा', 'Canada', 'Kya- cluster'),
    ('अस्ट्रेलिया', 'Australia', 'Str- cluster — country'),
    ('न्युजिल्याण्ड', 'New Zealand', 'Nyu- + final nd'),
    ('स्विट्जरल्याण्ड', 'Switzerland', 'Sw- + tzr- cluster'),
    ('नर्वे', 'Norway', 'Final -e'),
    ('फिनल्याण्ड', 'Finland', 'Fn- cluster'),


# Mountains & Peaks
('धौलागिरि', 'Dhaulagiri', 'Dh- + complex suffix — 7th highest'),
('कञ्चनजङ्घा', 'Kanchenjunga', 'Conjunct ञ्च + ङ्घ — 3rd highest'),
('अन्नपूर्णा', 'Annapurna', 'nn- geminate + rna cluster'),
('मनास्लु', 'Manaslu', 'Final -slu cluster'),
('चो ओयु', 'Cho Oyu', 'Tibetan — vowel sequence'),
('गणेश हिमाल', 'Ganesh Himal', 'Deity name + Himal'),
('लाङटाङ हिमाल', 'Langtang Himal', 'Ng-final + Tibetan'),
('त्रिशूल', 'Trishul', 'Tri- + shul — deity weapon/peak'),
('पुम्री हिउँचुली', 'Pumri Hiunchuli', 'Hiun + chuli — rare cluster'),
('थोरोङ ला', 'Thorong La', 'Tho- + Tibetan suffix La (pass)'),

# Rivers & Lakes
('त्रिशूलगंगा', 'Trishulganga', 'Tri- + compound + nasal'),
('भेरी नदी', 'Bheri Nadi', 'Bh- + nasal initial'),
('कर्णाली नदी', 'Karnali Nadi', 'rn- cluster — major river'),
('सप्तकोशी', 'Saptakoshi', 'Sapta + koshi compound'),
('त्रियुगा', 'Triyuga', 'Tri- + yuga — Udaypur river'),
('फेवा ताल', 'Phewa Tal', 'Ph- + Pokhara lake'),
('रारा ताल', 'Rara Tal', 'Reduplication — largest lake'),
('तिलिचो ताल', 'Tilicho Tal', 'High altitude — lch cluster'),
('गोसाइँकुण्ड', 'Gosainkunda', 'Compound — nd final + sacred lake'),
('बुढीगण्डकी', 'Budhigandaki', 'Bh- + ndak compound'),

# Historical / Heritage sites
('पशुपतिनाथ', 'Pashupatinatth', 'Sh- + compound — UNESCO site'),
('चाँगुनारायण', 'Changunnarayan', 'Chh- + compound — oldest temple'),
('न्यातापोल', 'Nyatapol', 'Ny- cluster — Bhaktapur'),
('भीमसेन थान', 'Bhimsen Than', 'Bh- + deity name'),
('हिरण्यवर्ण महाविहार', 'Hiranyavarna Mahavihara', 'rny- cluster — Golden Temple Patan'),
('कृष्णमन्दिर', 'Krishnamandir', 'Kr- + compound — Patan Durbar'),
('तुम्बहल', 'Tumbahall', 'mb- cluster — Newari courtyard'),
('वज्रयोगिनी', 'Bajrayogini', 'Vj- cluster + yogini'),

# Foreign countries (extended)
('पोर्चुगल', 'Portugal', 'Port- + final l'),
('स्पेन', 'Spain', 'Sp- cluster'),
('ग्रीस', 'Greece', 'Gr- cluster'),
('श्रीलङ्का', 'Sri Lanka', 'Shr- + conjunct ङ्क'),
('म्यानमार', 'Myanmar', 'My- cluster + final r'),
('थाइल्याण्ड', 'Thailand', 'Th- + final nd'),
('भियतनाम', 'Vietnam', 'Bh- + final m'),
('इन्डोनेसिया', 'Indonesia', 'nd- + vowel sequence'),
('अर्जेन्टिना', 'Argentina', 'rj- cluster + final a'),
('मेक्सिको', 'Mexico', 'ksi + o-final'),
('इथियोपिया', 'Ethiopia', 'Eth- + vowel sequence'),
('नाइजेरिया', 'Nigeria', 'Nai- diphthong + eria'),

# Person names (notable / rare clusters)
('पृथ्वीनारायण', 'Prithvinarayan', 'Pr- + thwi cluster — unifier king'),
('भानुभक्त', 'Bhanubhakta', 'Bh- + bh- + kt cluster — Aadikavi'),
('लक्ष्मीप्रसाद देवकोटा', 'Laxmiprasad Devkota', 'Lk- + compound + kota — Mahakavi'),
('माधवप्रसाद घिमिरे', 'Madhavprasad Ghimire', 'dh- + compound + Gh- — national poet'),
('तुलसीमेहर', 'Tulsimeher', 'ls- cluster + meher'),
('श्रद्धा', 'Shraddha', 'Shr- + ddha cluster — common female name'),
('प्रज्ञा', 'Pragya', 'Pr- + jñ- cluster — common female name'),
('ज्ञानेन्द्र', 'Gyanendra', 'Jñ- cluster — last king'),
('त्रिलोकचन्द्र', 'Trilokchandra', 'Tri- + lk + chandra compound'),
('स्वर्णलक्ष्मी', 'Swarnalaxmi', 'Sw- + rna + lk cluster — female name'),


# Ethnic / indigenous group names
('तामाङ', 'Tamang', 'Ng-final — major ethnic group'),
('नेवार', 'Newar', 'Final r — Kathmandu valley indigenous'),
('थारू', 'Tharu', 'Th- + u-final — Terai indigenous'),
('राई', 'Rai', 'Short — eastern hills ethnic'),
('लिम्बु', 'Limbu', 'mb- cluster — Kirat group'),
('मगर', 'Magar', 'Final r — western hills ethnic'),
('गुरुङ', 'Gurung', 'Ng-final — Annapurna region'),
('शेर्पा', 'Sherpa', 'Sh- + rpa cluster — Himalayan'),
('चेपाङ', 'Chepang', 'Ng-final — endangered group'),
('राजवंशी', 'Rajbanshi', 'Compound + sh- — Terai group'),
('ध्रुव', 'Dhruv', 'Dhr- + final v — rare cluster name'),
('भोटे', 'Bhote', 'Bh- + e-final — Tibetan-origin'),

# Sacred / pilgrimage sites
('हलेसी महादेव', 'Halesi Mahadev', 'Compound — major Shiva temple east'),
('पाथिभरा देवी', 'Pathibhara Devi', 'Bh- + ra — Taplejung temple'),
('मनकामना', 'Manakamana', 'mk- cluster — cable car temple'),
('बराहक्षेत्र', 'Barahakshetra', 'rh- + ksh- cluster — Sunsari'),
('देवघाट', 'Devghat', 'Compound — Chitwan confluence'),
('मुक्तिनाथ', 'Muktinath', 'kt- cluster — Mustang'),
('स्वर्गद्वारी', 'Swargadwari', 'Sw- + dw- — Pyuthan temple'),
('खप्तड', 'Khaptad', 'Kh- + retroflex — far-west'),

# Mountain passes & trekking routes
('थोरोङ पास', 'Thorong Pass', 'Tho- + ng-final + English'),
('तिल्मान पास', 'Tilman Pass', 'Tilman — lm cluster'),
('नाम्चे बजार', 'Namche Bazar', 'Nm- + Sherpa name'),
('फाप्लु', 'Phaplu', 'Ph- + plu cluster'),
('लुक्ला', 'Lukla', 'lk- cluster — famous airport'),
('गोक्यो री', 'Gokyo Ri', 'Gokyo — Tibetan + Ri (peak)'),
('कालापत्थर', 'Kalapathar', 'Compound — black rock viewpoint'),

# Terai / Madhesh cities & sites
('सप्तरी', 'Saptari', 'Sapt- + final i — district'),
('सिरहा', 'Siraha', 'rh- cluster — district'),
('महोत्तरी', 'Mahottari', 'Compound — central Terai district'),
('रौतहट', 'Rautahat', 'Raut- + hat — district'),
('नवलपुर', 'Nawalpur', 'Final r — recently renamed district'),
('पर्सा', 'Parsa', 'Pr- — Terai district'),
('चितवन', 'Chitwan', 'Chitw- cluster — famous national park'),
('कपिलवस्तु', 'Kapilvastu', 'Compound + stu — Buddha birthplace district'),

# Foreign cities (Nepali script, high-frequency in news)
('बेइजिङ', 'Beijing', 'Bei- + jing — Chinese capital'),
('वाशिङटन', 'Washington', 'Wh- + final n'),
('लन्डन', 'London', 'Final nd — British capital'),
('दिल्ली', 'Delhi', 'dh- + final i — Indian capital'),
('काठमाडौं', 'Kathmandu', 'Kth- + final nasal — own capital variant'),
],

'NAV': [
    ('बानेश्वर चोक', 'Baneshwor Chowk', 'Shw- cluster — Kathmandu junction'),
    ('बौद्ध स्तूप', 'Bouddha Stupa', 'Dh- + St- cluster'),
    ('स्वयम्भूनाथ', 'Swayambhunath', 'Sw- + conjunct bh'),
    ('ठमेल', 'Thamel', 'Th- aspirate — tourist hub'),
    ('पुतलीसडक', 'Putalisadak', 'Long compound — road name'),
    ('कोटेश्वर', 'Koteshwor', 'Shw- cluster — ring road junction'),
    ('चाबहिल', 'Chabahil', 'Aspirate — locality'),
    ('कालङ्की चोक', 'Kalanki Chowk', 'Lk- cluster — western junction'),
    ('त्रिपुरेश्वर', 'Tripureshwor', 'Tri- + shw- cluster'),
    ('बागबजार', 'Bagbazar', 'Compound — market'),
    ('दिल्लीबजार', 'Dillibazar', 'Borrowed (Delhi) + bazar'),
    ('लाजिम्पाट', 'Lazimpat', 'Final consonant cluster'),
    ('पुल्चोक', 'Pulchowk', 'Lch- cluster'),
    ('बालाजु बाइपास', 'Balaju Bypass', 'Bypass is English'),
    ('सिनामङ्गल', 'Sinamangal', 'Final consonant cluster'),
    ('नक्साल', 'Naxal', 'Final xal cluster'),
    ('माइतीघर', 'Maitighar', 'Diphthong + ghar'),
    ('रत्नपार्क', 'Ratnapark', 'Sanskrit + English'),
    ('घण्टाघर', 'Ghantanghar', 'Nasal cluster + ghar'),
    ('नयाँबानेश्वर', 'Naya Baneshwor', 'Shw- cluster — new area'),
    ('सूर्यविनायक', 'Suryabinayak', 'Compound — deity name + place'),
    ('भक्तपुर दरबार', 'Bhaktapur Durbar', 'Heritage site — Bhk- cluster'),
    ('ललितपुर महानगर', 'Lalitpur Mahanagar', 'Long compound'),
    ('किर्तिपुर', 'Kirtipur', 'Historical city — rt- cluster'),
    ('दक्षिणकाली', 'Dakshinkali', 'Dksh- cluster — temple'),
    ('चन्द्रागिरि', 'Chandragiri', 'Cable car hill — dr- cluster'),
    ('फर्पिङ', 'Pharping', 'Ph- + ng final'),
    ('नागार्जुन वन', 'Nagarjun Ban', 'Forest — Sanskrit name'),
    ('शिवपुरी', 'Shivapuri', 'Sh- + Sanskrit'),
    ('टोखा', 'Tokha', 'Retroflex T + aspirate'),
    ('बुढानीलकण्ठ', 'Budhanilkantha', 'Long compound — sleeping Vishnu'),
    ('जोरपाटी', 'Jorpati', 'Compound — road name'),
    ('गोकर्णेश्वर', 'Gokarneshwor', 'Shw- cluster — municipality'),
    ('साँखु', 'Sankhu', 'Kh- + ancient trade route'),
    ('नलु', 'Nalu', 'Short — village near Bhaktapur'),
    ('धुलिखेल', 'Dhulikhel', 'Dh- + lkh cluster'),
    ('पनौती', 'Panauti', 'Diphthong auti — heritage town'),
    ('बनेपा', 'Banepa', 'Historical town — np- cluster'),
    ('काभ्रेपलाञ्चोक', 'Kavrepalanchok', 'District — conjunct ञ्च'),
    ('सिन्धुली', 'Sindhuli', 'District — ndh- cluster'),
    ('रामेछाप', 'Ramechhap', 'District — chh aspirate'),
    ('ओखलढुङ्गा', 'Okhaldhunga', 'District — kh + dh + conjunct'),
    ('सोलुखुम्बु', 'Solukhumbu', 'District — kh + mb cluster'),
    ('खोटाङ सदरमुकाम', 'Khotang Sadar', 'Kh- + compound'),
    ('तेह्रथुम', 'Tehrathum', 'District — thr- cluster'),
    ('ताप्लेजुङ', 'Taplejung', 'District — ng-final'),
    ('पाँचथर', 'Panchthar', 'District — thr- cluster'),
    ('इलाम बजार', 'Ilam Bazar', 'Initial vowel + compound'),
    ('धनकुटा', 'Dhankuta', 'Dh- + retroflex'),
    ('भोजपुर', 'Bhojpur', 'Bh- + pur suffix'),
    ('खाँदबारी', 'Khandabari', 'Kh- + nasal + retroflex'),
    ('फिदिम', 'Phidim', 'Ph- + dim'),
    ('हेटौँडा', 'Hetauda', 'Industrial city — ou diphthong'),
    ('बुटवल', 'Butwal', 'Western hub — final l'),
    ('भैरहवा', 'Bhairahawa', 'Bh- + rahawa'),
    ('नेपालगञ्ज', 'Nepalgunj', 'City — final conjunct ञ्ज'),
    ('धनगढी', 'Dhangadhi', 'Far-west city — Dh- + dh'),
    ('महेन्द्रनगर', 'Mahendranagar', 'Far-west city — compound'),
    ('दमक', 'Damak', 'Eastern town — simple'),
    ('बिर्तामोड', 'Birtamod', 'Eastern hub — b/v + final d'),
    ('इटहरी', 'Itahari', 'Eastern city — initial vowel'),
    ('ढल्केबर', 'Dhalkebar', 'Dh- + retroflex cluster'),
    ('जलेश्वर', 'Jaleshwor', 'Shw- cluster — Madhesh city'),
    ('जनकपुर धाम', 'Janakpur Dham', 'Religious city — dham suffix'),
    ('राजविराज', 'Rajbiraj', 'Madhesh capital — compound'),
    ('लहान', 'Lahan', 'Madhesh town — simple'),
    ('गौर', 'Gaur', 'Rautahat HQ — au diphthong'),
    ('कलैया', 'Kalaiya', 'Bara HQ — ai diphthong'),
    ('वीरगञ्ज', 'Birgunj', 'Trade hub — conjunct ञ्ज'),
    ('सिमरा', 'Simara', 'Airport town — simple'),
    ('पोखरा लेकसाइड', 'Pokhara Lakeside', 'Lakeside — code-mix + English'),
     ('ल्हासा',       'Lhasa',       'Lh- cluster — Tibetan'),
    ('ल्होत्से',     'Lhotse',      'Lh- cluster — mountain'),
    ('मकालु',        'Makalu',      'Borrowed Tibetan/Sanskrit'),
    ('मुस्ताङ',      'Mustang',     'Ng-final — region name'),
    ('खोटाङ',        'Khotang',     'Kh- + nasal final'),
    ('सोलुखुम्बु',   'Solukhumbu',  'Kh- cluster + labial'),
    ('सिन्धुपाल्चोक','Sindhupalchok','Conjunct ल्च'),
    ('दोलखा',        'Dolkha',      'Lkh cluster'),
    ('मनाङ',         'Manang',      'Ng-final'),
    ('डोल्पा',       'Dolpa',       'Retroflex D + final a'),
    ('ह्युम्बु',     'Hyumbu',      'Hy- + Tibetan origin'),
    ('त्सोरोलपा',    'Tso Rolpa',   'Tso- glacier lake — Tibetan'),
    ('फ्वाङलुङ',     'Phwanglung',  'Phw- cluster — Tibetan place'),
    ('ग्ल्यासिअर',   'Glacier',     'English loanword as place reference'),
    ('ज्वालामुखी',   'Jwalamukhi',  'Jw- cluster — volcanic'),
    ('श्रीस्वस्थानी', 'Shree Swasthani','Shree + sw- cluster'),
    ('फ्रान्स',       'France',      'Fr- cluster — country name'),
    ('ब्राजिल',       'Brazil',      'Br- + final l'),
    ('जर्मनी',        'Germany',     'Jrm- cluster'),
    ('क्यानडा',       'Canada',      'Kya- cluster'),
    ('अस्ट्रेलिया',   'Australia',   'Str- cluster — country'),
    ('न्युजिल्याण्ड', 'New Zealand', 'Nyu- + final nd'),
    ('स्विट्जरल्याण्ड','Switzerland','Sw- + tzr- cluster'),
    ('नर्वे',          'Norway',      'Final -e'),
    ('फिनल्याण्ड',    'Finland',     'Fn- cluster'),

],

'NAV': [
    ('बानेश्वर चोक', 'Baneshwor Chowk', 'Shw- cluster — Kathmandu junction'),
    ('बौद्ध स्तूप', 'Bouddha Stupa', 'Dh- + St- cluster'),
    ('स्वयम्भूनाथ', 'Swayambhunath', 'Sw- + conjunct bh'),
    ('ठमेल', 'Thamel', 'Th- aspirate — tourist hub'),
    ('पुतलीसडक', 'Putalisadak', 'Long compound — road name'),
    ('कोटेश्वर', 'Koteshwor', 'Shw- cluster — ring road junction'),
    ('चाबहिल', 'Chabahil', 'Aspirate — locality'),
    ('कालङ्की चोक', 'Kalanki Chowk', 'Lk- cluster — western junction'),
    ('त्रिपुरेश्वर', 'Tripureshwor', 'Tri- + shw- cluster'),
    ('बागबजार', 'Bagbazar', 'Compound — market'),
    ('दिल्लीबजार', 'Dillibazar', 'Borrowed (Delhi) + bazar'),
    ('लाजिम्पाट', 'Lazimpat', 'Final consonant cluster'),
    ('पुल्चोक', 'Pulchowk', 'Lch- cluster'),
    ('बालाजु बाइपास', 'Balaju Bypass', 'Bypass is English'),
    ('सिनामङ्गल', 'Sinamangal', 'Final consonant cluster'),
    ('नक्साल', 'Naxal', 'Final xal cluster'),
    ('माइतीघर', 'Maitighar', 'Diphthong + ghar'),
    ('रत्नपार्क', 'Ratnapark', 'Sanskrit + English'),
    ('घण्टाघर', 'Ghantanghar', 'Nasal cluster + ghar'),
    ('नयाँबानेश्वर', 'Naya Baneshwor', 'Shw- cluster — new area'),
    ('सूर्यविनायक', 'Suryabinayak', 'Compound — deity name + place'),
    ('भक्तपुर दरबार', 'Bhaktapur Durbar', 'Heritage site — Bhk- cluster'),
    ('ललितपुर महानगर', 'Lalitpur Mahanagar', 'Long compound'),
    ('किर्तिपुर', 'Kirtipur', 'Historical city — rt- cluster'),
    ('दक्षिणकाली', 'Dakshinkali', 'Dksh- cluster — temple'),
    ('चन्द्रागिरि', 'Chandragiri', 'Cable car hill — dr- cluster'),
    ('फर्पिङ', 'Pharping', 'Ph- + ng final'),
    ('नागार्जुन वन', 'Nagarjun Ban', 'Forest — Sanskrit name'),
    ('शिवपुरी', 'Shivapuri', 'Sh- + Sanskrit'),
    ('टोखा', 'Tokha', 'Retroflex T + aspirate'),
    ('बुढानीलकण्ठ', 'Budhanilkantha', 'Long compound — sleeping Vishnu'),
    ('जोरपाटी', 'Jorpati', 'Compound — road name'),
    ('गोकर्णेश्वर', 'Gokarneshwor', 'Shw- cluster — municipality'),
    ('साँखु', 'Sankhu', 'Kh- + ancient trade route'),
    ('नलु', 'Nalu', 'Short — village near Bhaktapur'),
    ('धुलिखेल', 'Dhulikhel', 'Dh- + lkh cluster'),
    ('पनौती', 'Panauti', 'Diphthong auti — heritage town'),
    ('बनेपा', 'Banepa', 'Historical town — np- cluster'),
    ('काभ्रेपलाञ्चोक', 'Kavrepalanchok', 'District — conjunct ञ्च'),
    ('सिन्धुली', 'Sindhuli', 'District — ndh- cluster'),
    ('रामेछाप', 'Ramechhap', 'District — chh aspirate'),
    ('ओखलढुङ्गा', 'Okhaldhunga', 'District — kh + dh + conjunct'),
    ('सोलुखुम्बु', 'Solukhumbu', 'District — kh + mb cluster'),
    ('खोटाङ सदरमुकाम', 'Khotang Sadar', 'Kh- + compound'),
    ('तेह्रथुम', 'Tehrathum', 'District — thr- cluster'),
    ('ताप्लेजुङ', 'Taplejung', 'District — ng-final'),
    ('पाँचथर', 'Panchthar', 'District — thr- cluster'),
    ('इलाम बजार', 'Ilam Bazar', 'Initial vowel + compound'),
    ('धनकुटा', 'Dhankuta', 'Dh- + retroflex'),
    ('भोजपुर', 'Bhojpur', 'Bh- + pur suffix'),
    ('खाँदबारी', 'Khandabari', 'Kh- + nasal + retroflex'),
    ('फिदिम', 'Phidim', 'Ph- + dim'),
    ('हेटौँडा', 'Hetauda', 'Industrial city — ou diphthong'),
    ('बुटवल', 'Butwal', 'Western hub — final l'),
    ('भैरहवा', 'Bhairahawa', 'Bh- + rahawa'),
    ('नेपालगञ्ज', 'Nepalgunj', 'City — final conjunct ञ्ज'),
    ('धनगढी', 'Dhangadhi', 'Far-west city — Dh- + dh'),
    ('महेन्द्रनगर', 'Mahendranagar', 'Far-west city — compound'),
    ('दमक', 'Damak', 'Eastern town — simple'),
    ('बिर्तामोड', 'Birtamod', 'Eastern hub — b/v + final d'),
    ('इटहरी', 'Itahari', 'Eastern city — initial vowel'),
    ('ढल्केबर', 'Dhalkebar', 'Dh- + retroflex cluster'),
    ('जलेश्वर', 'Jaleshwor', 'Shw- cluster — Madhesh city'),
    ('जनकपुर धाम', 'Janakpur Dham', 'Religious city — dham suffix'),
    ('राजविराज', 'Rajbiraj', 'Madhesh capital — compound'),
    ('लहान', 'Lahan', 'Madhesh town — simple'),
    ('गौर', 'Gaur', 'Rautahat HQ — au diphthong'),
    ('कलैया', 'Kalaiya', 'Bara HQ — ai diphthong'),
    ('वीरगञ्ज', 'Birgunj', 'Trade hub — conjunct ञ्ज'),
    ('सिमरा', 'Simara', 'Airport town — simple'),
    ('पोखरा लेकसाइड', 'Pokhara Lakeside', 'Lakeside — code-mix + English'),
    ('बानेश्वर चोक',    'Baneshwor Chowk',  'Shw- cluster — Kathmandu junction'),
    ('बौद्ध स्तूप',     'Bouddha Stupa',    'Dh- + St- cluster'),
    ('स्वयम्भूनाथ',     'Swayambhunath',    'Sw- + conjunct bh'),
    ('ठमेल',            'Thamel',           'Th- aspirate — tourist hub'),
    ('पुतलीसडक',        'Putalisadak',      'Long compound — road name'),
    ('कोटेश्वर',        'Koteshwor',        'Shw- cluster — ring road junction'),
    ('चाबहिल',          'Chabahil',         'Aspirate — locality'),
    ('कालङ्की चोक',     'Kalanki Chowk',    'Lk- cluster — western junction'),
    ('त्रिपुरेश्वर',    'Tripureshwor',     'Tri- + shw- cluster'),
    ('बागबजार',         'Bagbazar',         'Compound — market'),
    ('दिल्लीबजार',      'Dillibazar',       'Borrowed (Delhi) + bazar'),
    ('लाजिम्पाट',       'Lazimpat',         'Final consonant cluster'),
    ('पुल्चोक',         'Pulchowk',         'Lch- cluster'),
    ('बालाजु बाइपास',   'Balaju Bypass',    'Bypass is English'),
    ('सिनामङ्गल',       'Sinamangal',       'Final consonant cluster'),
    ('नक्साल',          'Naxal',            'Final xal cluster'),
    ('माइतीघर',         'Maitighar',        'Diphthong + ghar'),
    ('रत्नपार्क',       'Ratnapark',        'Sanskrit + English'),
    ('घण्टाघर',         'Ghantanghar',      'Nasal cluster + ghar'),
    ('नयाँबानेश्वर',    'Naya Baneshwor',   'Shw- cluster — new area'),
    ('सूर्यविनायक',     'Suryabinayak',     'Compound — deity name + place'),
    ('भक्तपुर दरबार',   'Bhaktapur Durbar', 'Heritage site — Bhk- cluster'),
    ('ललितपुर महानगर',  'Lalitpur Mahanagar','Long compound'),
    ('किर्तिपुर',       'Kirtipur',         'Historical city — rt- cluster'),
    ('दक्षिणकाली',      'Dakshinkali',      'Dksh- cluster — temple'),
    ('चन्द्रागिरि',     'Chandragiri',      'Cable car hill — dr- cluster'),
    ('फर्पिङ',          'Pharping',         'Ph- + ng final'),
    ('नागार्जुन वन',    'Nagarjun Ban',     'Forest — Sanskrit name'),
    ('शिवपुरी',         'Shivapuri',        'Sh- + Sanskrit'),
    ('टोखा',            'Tokha',            'Retroflex T + aspirate'),
    ('बुढानीलकण्ठ',     'Budhanilkantha',   'Long compound — sleeping Vishnu'),
    ('जोरपाटी',         'Jorpati',          'Compound — road name'),
    ('गोकर्णेश्वर',     'Gokarneshwor',     'Shw- cluster — municipality'),
    ('साँखु',           'Sankhu',           'Kh- + ancient trade route'),
    ('नलु',             'Nalu',             'Short — village near Bhaktapur'),
    ('धुलिखेल',         'Dhulikhel',        'Dh- + lkh cluster'),
    ('पनौती',           'Panauti',          'Diphthong auti — heritage town'),
    ('बनेपा',           'Banepa',           'Historical town — np- cluster'),
    ('काभ्रेपलाञ्चोक',  'Kavrepalanchok',   'District — conjunct ञ्च'),
    ('सिन्धुली',        'Sindhuli',         'District — ndh- cluster'),
    ('रामेछाप',         'Ramechhap',        'District — chh aspirate'),
    ('ओखलढुङ्गा',       'Okhaldhunga',      'District — kh + dh + conjunct'),
    ('सोलुखुम्बु',      'Solukhumbu',       'District — kh + mb cluster'),
    ('खोटाङ सदरमुकाम',  'Khotang Sadar',    'Kh- + compound'),
    ('तेह्रथुम',        'Tehrathum',        'District — thr- cluster'),
    ('ताप्लेजुङ',       'Taplejung',        'District — ng-final'),
    ('पाँचथर',          'Panchthar',        'District — thr- cluster'),
    ('इलाम बजार',       'Ilam Bazar',       'Initial vowel + compound'),
    ('धनकुटा',          'Dhankuta',         'Dh- + retroflex'),
    ('भोजपुर',          'Bhojpur',          'Bh- + pur suffix'),
    ('खाँदबारी',        'Khandabari',       'Kh- + nasal + retroflex'),
    ('फिदिम',           'Phidim',           'Ph- + dim'),
],

}

# Quick count
print("Seed OOV words per category:")
for cat, words in SEED_OOV.items():
    print(f"  {cat:6s} : {len(words):3d} seed words")


Seed OOV words per category:
  ABBR   : 151 seed words
  BRAND  : 137 seed words
  CM     : 135 seed words
  CMPY   : 128 seed words
  GOVT   : 123 seed words
  PROP   : 115 seed words
  NAV    : 123 seed words


In [ ]:
# Merge seed words into per-category DataFrames
# Only adds seeds that are not already present in step4 CSVs.

merged = {}
for cat in CATEGORIES:
    existing = cat_dfs[cat].copy()
    existing_words = set(existing['word'].str.strip().tolist())

    seeds = SEED_OOV.get(cat, [])
    new_rows = []
    for entry in seeds:
        word = entry[0].strip()
        notes = entry[2] if len(entry) > 2 else ''
        if word not in existing_words:
            new_rows.append({
                'word':     word,
                'category': cat,
                'is_oov':   True,
                'notes':    notes,
                'source':   'seed_guide'
            })
            existing_words.add(word)

    if new_rows:
        seed_df = pd.DataFrame(new_rows)
        if 'source' not in existing.columns:
            existing['source'] = 'step4_corpus'
        combined = pd.concat([existing, seed_df], ignore_index=True)
    else:
        if 'source' not in existing.columns:
            existing['source'] = 'step4_corpus'
        combined = existing

    merged[cat] = combined
    n_total    = len(combined)
    n_new      = len(new_rows)
    status     = '✓' if n_total >= TARGET_OOV else f'⚠ need {TARGET_OOV - n_total} more'
    print(f"  {cat:6s} : {n_total:4d} OOV total  (+{n_new:3d} seeds)  {status}")


  ABBR   :   75 OOV total  (+ 75 seeds)  ✓
  BRAND  :   75 OOV total  (+ 75 seeds)  ✓
  CM     :   75 OOV total  (+ 75 seeds)  ✓
  CMPY   :   75 OOV total  (+ 74 seeds)  ✓
  GOVT   :   75 OOV total  (+ 73 seeds)  ✓
  PROP   : 2108 OOV total  (+115 seeds)  ✓
  NAV    :   75 OOV total  (+ 71 seeds)  ✓


In [ ]:
import json
from pathlib import Path

# Load TTS training IV bigrams and vocabulary
# Using combined_tts_iv_FINAL.json (Step 1 output)

IV_FINAL_PATH = Path('combined_tts_iv_FINAL.json')

if IV_FINAL_PATH.exists():
    with open(IV_FINAL_PATH, encoding='utf-8') as f:
        iv_data = json.load(f)
    iv_bigrams   = set(iv_data['iv_bigrams'])
    iv_words_all = iv_data['iv_words']
    print(f'Loaded from {IV_FINAL_PATH}')
    print(f'Sources    : {iv_data["sources"]}')
else:
    raise FileNotFoundError(
        'combined_tts_iv_FINAL.json not found. '
        'Upload it to the same directory as this notebook.'
    )

print(f'IV bigrams : {len(iv_bigrams):,}')
print(f'IV words   : {len(iv_words_all):,}')

# Helper: confirm a word is truly IV (all bigrams present)
def is_iv(word: str) -> bool:
    if len(word) < 2:
        return False
    bgs = {word[i:i+2] for i in range(len(word)-1)}
    return bgs.issubset(iv_bigrams)

# Pre-filter IV pool
iv_pool = [w for w in iv_words_all if is_iv(w) and len(w) >= 3]
print(f'IV pool after filter : {len(iv_pool):,}')

Loaded from combined_tts_iv_FINAL.json
Sources    : ['Titung/nepali-tts-tagged-combined', 'Titung/cc100-nepali-tts']
IV bigrams : 2,348
IV words   : 70,537
IV pool after filter : 70,131


In [ ]:
import re

LATIN_RE    = re.compile(r'[A-Za-z]')
ALL_CAPS_RE = re.compile(r'^[A-Z]{2,}$')
DEVA_CHAR_RE = re.compile(r'[\u0900-\u097F]')   # ← fixed: search for ANY Devanagari char
DEVA_ONLY_RE = re.compile(r'^[\u0900-\u097F\s]+$')

PLACE_SUFFIXES  = ('गर', 'पुर', 'गञ्ज', 'नगर', 'बजार', 'चोक', 'टोल',
                   'खोला', 'दह', 'ताल', 'डाँडा', 'गाउँ', 'थान', 'कुण्ड')
PERSON_SUFFIXES = ('लाल', 'बहादुर', 'प्रसाद', 'कुमार', 'देवी', 'माया',
                   'राज', 'मान', 'धर', 'दास')
GOVT_KEYWORDS   = ('योजना', 'कार्यक्रम', 'आयोग', 'समिति', 'बोर्ड',
                   'मन्त्रालय', 'विभाग', 'कोष', 'निगम', 'प्राधिकरण')
CMPY_KEYWORDS   = ('बैंक', 'इन्स्योरेन्स', 'फाइनान्स', 'ग्रुप',
                   'लिमिटेड', 'कर्पोरेशन', 'इन्टरनेशनल', 'नेपाल')

def classify_iv(word: str) -> str:
    if ALL_CAPS_RE.match(word):                             return 'ABBR'
    # CM: has BOTH Latin chars and Devanagari chars
    if LATIN_RE.search(word) and DEVA_CHAR_RE.search(word): return 'CM'
    # BRAND: Latin only (no Devanagari)
    if LATIN_RE.search(word):                               return 'BRAND'
    if any(kw in word for kw in GOVT_KEYWORDS):             return 'GOVT'
    if any(kw in word for kw in CMPY_KEYWORDS):             return 'CMPY'
    if any(word.endswith(s) for s in PLACE_SUFFIXES):       return 'PROP'
    if any(word.endswith(s) for s in PERSON_SUFFIXES):      return 'PROP'
    if any(kw in word for kw in ('चोक','मार्ग','सडक','रोड','पथ')): return 'NAV'
    return 'PROP'

# Build per-category IV pools
iv_by_cat = {cat: [] for cat in CATEGORIES}
for w in iv_pool:
    iv_by_cat[classify_iv(w)].append(w)

# For Latin-only categories, IV candidates must come from seed words
# ABBR, BRAND, CM seeds are already Latin/mixed — they ARE the pool for those cats.
# The greedy selection step will just use seeds directly for these three.
print('IV pool per category:')
for cat in CATEGORIES:
    n_iv    = len(iv_by_cat[cat])
    n_seeds = len(SEED_OOV.get(cat, []))
    note = '  ← seeds only (Latin-script, no IV pool)' if n_iv == 0 else ''
    print(f'  {cat:6s} : {n_iv:5d} candidates  ({n_seeds} seeds){note}')

IV pool per category:
  ABBR   :     0 candidates  (151 seeds)  ← seeds only (Latin-script, no IV pool)
  BRAND  :     0 candidates  (137 seeds)  ← seeds only (Latin-script, no IV pool)
  CM     :     0 candidates  (135 seeds)  ← seeds only (Latin-script, no IV pool)
  CMPY   :   224 candidates  (128 seeds)
  GOVT   :   333 candidates  (123 seeds)
  PROP   : 69306 candidates  (115 seeds)
  NAV    :   268 candidates  (123 seeds)


In [ ]:
# Sample IV words per category
# For thin IV pools (ABBR/BRAND/CM), fall back to PROP pool.

iv_dfs = {}
for cat in CATEGORIES:
    pool = iv_by_cat[cat][:]
    # Fallback: draw from PROP pool if category is too thin
    if len(pool) < TARGET_IV:
        extra_needed = TARGET_IV - len(pool)
        fallback = [w for w in iv_by_cat['PROP'] if w not in pool]
        random.shuffle(fallback)
        pool = pool + fallback[:extra_needed]
        print(f'  {cat:6s} IV pool thin — drew {extra_needed} extra from PROP fallback')

    sampled = random.sample(pool, min(TARGET_IV, len(pool)))
    iv_df = pd.DataFrame({
        'word':     sampled,
        'category': cat,
        'is_oov':   False,
        'notes':    'IV control word — all bigrams in training vocab',
        'source':   'iv_sample'
    })
    iv_dfs[cat] = iv_df
    print(f'  {cat:6s} : {len(iv_df):3d} IV words sampled')


  ABBR   IV pool thin — drew 75 extra from PROP fallback
  ABBR   :  75 IV words sampled
  BRAND  IV pool thin — drew 75 extra from PROP fallback
  BRAND  :  75 IV words sampled
  CM     IV pool thin — drew 75 extra from PROP fallback
  CM     :  75 IV words sampled
  CMPY   :  75 IV words sampled
  GOVT   :  75 IV words sampled
  PROP   :  75 IV words sampled
  NAV    :  75 IV words sampled


In [ ]:
os.makedirs('step4b_results/by_category', exist_ok=True)

print('=' * 65)
print(f"{'Category':8s}  {'OOV':>6s}  {'IV':>6s}  {'Total':>7s}  {'OOV≥75':>7s}  {'IV≥75':>6s}")
print('-' * 65)

all_ready = True
for cat in CATEGORIES:
    oov_df = merged[cat].copy()
    oov_df['is_oov'] = True

    iv_df  = iv_dfs[cat].copy()
    iv_df['is_oov'] = False

    final  = pd.concat([oov_df, iv_df], ignore_index=True)
    # Ensure required columns in correct order
    for col in ['word','category','is_oov','notes','source']:
        if col not in final.columns:
            final[col] = ''
    final  = final[['word','category','is_oov','notes','source']]

    out = f'step4b_results/by_category/{cat}.csv'
    final.to_csv(out, index=False, encoding='utf-8-sig')

    n_oov = int((final['is_oov'] == True).sum())
    n_iv  = int((final['is_oov'] == False).sum())
    ok_oov = '✓' if n_oov >= TARGET_OOV else '✗'
    ok_iv  = '✓' if n_iv  >= TARGET_IV  else '✗'
    if ok_oov == '✗' or ok_iv == '✗':
        all_ready = False

    print(f'  {cat:6s}    {n_oov:6d}  {n_iv:6d}  {len(final):7d}  {ok_oov:>7s}  {ok_iv:>6s}')

print('=' * 65)
if all_ready:
    print('\n✅ All categories meet the ≥75 OOV + ≥75 IV target.')
    print('   Ready to proceed to Step 5 — Create Evaluation Sentences.')
else:
    print('\n⚠  Some categories are still below target.')
    print('   Review the table above and add more words manually.')


Category     OOV      IV    Total   OOV≥75   IV≥75
-----------------------------------------------------------------
  ABBR          75      75      150        ✓       ✓
  BRAND         75      75      150        ✓       ✓
  CM            75      75      150        ✓       ✓
  CMPY          75      75      150        ✓       ✓
  GOVT          75      75      150        ✓       ✓
  PROP        2108      75     2183        ✓       ✓
  NAV           75      75      150        ✓       ✓

✅ All categories meet the ≥75 OOV + ≥75 IV target.
   Ready to proceed to Step 5 — Create Evaluation Sentences.


In [ ]:
# Final readiness report

print('\n📋 STEP 4b READINESS REPORT')
print('=' * 50)

total_oov = 0
total_iv  = 0
for cat in CATEGORIES:
    df = pd.read_csv(f'step4b_results/by_category/{cat}.csv', encoding='utf-8-sig')
    n_oov = int((df['is_oov'] == True).sum())
    n_iv  = int((df['is_oov'] == False).sum())
    total_oov += n_oov
    total_iv  += n_iv
    by_src = df['source'].value_counts().to_dict()
    src_str = '  '.join(f"{k}:{v}" for k, v in by_src.items())
    print(f'  {cat:6s}  OOV={n_oov:3d}  IV={n_iv:3d}  | sources: {src_str}')

print('-' * 50)
print(f'  TOTAL   OOV={total_oov}  IV={total_iv}  Grand total={total_oov+total_iv}')
print()
print('Benchmark minimum check (50 OOV + 50 IV per category):')
for cat in CATEGORIES:
    df = pd.read_csv(f'step4b_results/by_category/{cat}.csv', encoding='utf-8-sig')
    n_oov = int((df['is_oov'] == True).sum())
    n_iv  = int((df['is_oov'] == False).sum())
    oov_ok = '✓' if n_oov >= 50 else f'✗ ({50-n_oov} short)'
    iv_ok  = '✓' if n_iv  >= 50 else f'✗ ({50-n_iv} short)'
    print(f'  {cat:6s}  OOV ≥ 50: {oov_ok:20s}  IV ≥ 50: {iv_ok}')

print()
print('Next step → Step 5: Create Evaluation Sentences')
print('  File: step4b_results/by_category/<CAT>.csv')
print('  Each row has: word, category, is_oov, notes, source')



📋 STEP 4b READINESS REPORT
  ABBR    OOV= 75  IV= 75  | sources: seed_guide:75  iv_sample:75
  BRAND   OOV= 75  IV= 75  | sources: seed_guide:75  iv_sample:75
  CM      OOV= 75  IV= 75  | sources: seed_guide:75  iv_sample:75
  CMPY    OOV= 75  IV= 75  | sources: iv_sample:75  seed_guide:74  step4_corpus:1
  GOVT    OOV= 75  IV= 75  | sources: iv_sample:75  seed_guide:73  step4_corpus:2
  PROP    OOV=2108  IV= 75  | sources: step4_corpus:1993  seed_guide:115  iv_sample:75
  NAV     OOV= 75  IV= 75  | sources: iv_sample:75  seed_guide:71  step4_corpus:4
--------------------------------------------------
  TOTAL   OOV=2558  IV=525  Grand total=3083

Benchmark minimum check (50 OOV + 50 IV per category):
  ABBR    OOV ≥ 50: ✓                     IV ≥ 50: ✓
  BRAND   OOV ≥ 50: ✓                     IV ≥ 50: ✓
  CM      OOV ≥ 50: ✓                     IV ≥ 50: ✓
  CMPY    OOV ≥ 50: ✓                     IV ≥ 50: ✓
  GOVT    OOV ≥ 50: ✓                     IV ≥ 50: ✓
  PROP    OOV ≥ 50: ✓   

In [28]:
# Optional — cap PROP to 150 OOV to balance recording load
# Skip this if you want the full 2108 PROP words in the benchmark

PROP_CAP = 150

prop_csv = pd.read_csv('step4b_results/by_category/PROP.csv')
oov_rows = prop_csv[prop_csv['is_oov'] == True]
iv_rows  = prop_csv[prop_csv['is_oov'] == False]

if len(oov_rows) > PROP_CAP:
    # Prioritise seed_guide first, then step4_corpus
    seeds  = oov_rows[oov_rows['source'] == 'seed_guide']
    corpus = oov_rows[oov_rows['source'] == 'step4_corpus']
    capped = pd.concat([seeds, corpus]).head(PROP_CAP)
    prop_csv = pd.concat([capped, iv_rows])
    prop_csv.to_csv('step4b_results/by_category/PROP.csv', index=False)
    print(f'PROP capped: {len(oov_rows)} → {PROP_CAP} OOV rows')

PROP capped: 2108 → 150 OOV rows


In [ ]:
import json
import random
import csv
import os
from pathlib import Path
from collections import defaultdict

import pandas as pd

# Config 
SEED          = 42          # for reproducibility
GROUP_SIZE    = 5           # words per utterance (paper default)
PROP_OOV_CAP  = 150         # cap PROP OOV to match other categories; set None to keep all 2108

INPUT_DIR     = Path('step4b_results/by_category')
OUTPUT_DIR    = Path('step5_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = ['ABBR', 'BRAND', 'CM', 'CMPY', 'GOVT', 'PROP', 'NAV']

random.seed(SEED)
print('Config OK')
print(f'  GROUP_SIZE   : {GROUP_SIZE}')
print(f'  PROP_OOV_CAP : {PROP_OOV_CAP}')
print(f'  INPUT_DIR    : {INPUT_DIR}')
print(f'  OUTPUT_DIR   : {OUTPUT_DIR}')

Config OK
  GROUP_SIZE   : 5
  PROP_OOV_CAP : 150
  INPUT_DIR    : step4b_results/by_category
  OUTPUT_DIR   : step5_results


In [ ]:
all_words = []

for cat in CATEGORIES:
    csv_path = INPUT_DIR / f'{cat}.csv'
    if not csv_path.exists():
        print(f'  ⚠  {csv_path} not found — skipping')
        continue
    df = pd.read_csv(csv_path, encoding='utf-8')

    # Normalise is_oov to bool
    df['is_oov'] = df['is_oov'].astype(str).str.strip().str.lower().map(
        {'true': True, '1': True, 'false': False, '0': False}
    ).fillna(False)

    # Apply PROP OOV cap
    if cat == 'PROP' and PROP_OOV_CAP is not None:
        oov_rows = df[df['is_oov'] == True]
        iv_rows  = df[df['is_oov'] == False]
        if len(oov_rows) > PROP_OOV_CAP:
            seeds  = oov_rows[oov_rows['source'] == 'seed_guide']
            corpus = oov_rows[oov_rows['source'] == 'step4_corpus']
            capped = pd.concat([seeds, corpus]).head(PROP_OOV_CAP)
            df = pd.concat([capped, iv_rows]).reset_index(drop=True)
            print(f'  PROP OOV capped: {len(oov_rows)} → {PROP_OOV_CAP}')

    records = df.to_dict('records')
    all_words.extend(records)
    n_oov = df['is_oov'].sum()
    n_iv  = (~df['is_oov']).sum()
    print(f'  {cat:6s} : {len(df):4d} rows  (OOV={n_oov}, IV={n_iv})')

print(f'\nTotal words loaded : {len(all_words)}')

# Deduplicate: if a word appears as both OOV and IV, keep OOV only 
seen = {}
deduped = []
for r in all_words:
    w = r['word']
    if w not in seen:
        seen[w] = r
        deduped.append(r)
    else:
        # Keep OOV over IV if there's a conflict
        if r['is_oov'] and not seen[w]['is_oov']:
            seen[w] = r
            deduped = [r if x['word'] == w else x for x in deduped]

dupes_removed = len(all_words) - len(deduped)
all_words = deduped
if dupes_removed:
    print(f'\n⚠  Removed {dupes_removed} duplicate word(s) (kept OOV over IV)')
else:
    print('\n✓  No duplicates found')
print(f'Total after dedup  : {len(all_words)}')

  ABBR   :  150 rows  (OOV=75, IV=75)
  BRAND  :  150 rows  (OOV=75, IV=75)
  CM     :  150 rows  (OOV=75, IV=75)
  CMPY   :  150 rows  (OOV=75, IV=75)
  GOVT   :  150 rows  (OOV=75, IV=75)
  PROP   :  225 rows  (OOV=150, IV=75)
  NAV    :  150 rows  (OOV=75, IV=75)

Total words loaded : 1125

⚠  Removed 4 duplicate word(s) (kept OOV over IV)
Total after dedup  : 1121


In [ ]:
def make_utterances(word_records, group_size=5, seed=42):
# Shuffle word_records and group into utterances of `group_size`.
# Returns list of dicts with keys: utterance_id, words, sentence,
# categories, is_oov_flags.
# Leftover words (< group_size) are dropped to keep utterances uniform.
    rng = random.Random(seed)
    records = word_records.copy()
    rng.shuffle(records)

    utterances = []
    for i in range(0, len(records) - group_size + 1, group_size):
        group = records[i : i + group_size]
        utterances.append({
            'words'         : [r['word'] for r in group],
            'sentence'      : ', '.join(r['word'] for r in group),
            'categories'    : [r['category'] for r in group],
            'is_oov_flags'  : [r['is_oov'] for r in group],
            'sources'       : [r.get('source', '') for r in group],
        })
    return utterances


oov_records = [r for r in all_words if r['is_oov'] == True]
iv_records  = [r for r in all_words if r['is_oov'] == False]

oov_utterances = make_utterances(oov_records, GROUP_SIZE, seed=SEED)
iv_utterances  = make_utterances(iv_records,  GROUP_SIZE, seed=SEED + 1)

print(f'OOV words    : {len(oov_records):4d}  →  {len(oov_utterances):4d} utterances  '
      f'(dropped {len(oov_records) - len(oov_utterances)*GROUP_SIZE} leftover words)')
print(f'IV  words    : {len(iv_records):4d}  →  {len(iv_utterances):4d} utterances  '
      f'(dropped {len(iv_records)  - len(iv_utterances)*GROUP_SIZE}  leftover words)')
print(f'Total utterances : {len(oov_utterances) + len(iv_utterances)}')

OOV words    :  599  →   119 utterances  (dropped 4 leftover words)
IV  words    :  522  →   104 utterances  (dropped 2  leftover words)
Total utterances : 223


In [41]:
eval_rows = []
sentence_id = 1

for utt_list, label in [(oov_utterances, 'OOV'), (iv_utterances, 'IV')]:
    for utt in utt_list:
        sid = f'{label}_{sentence_id:04d}'
        for word, cat, is_oov, src in zip(
            utt['words'], utt['categories'], utt['is_oov_flags'], utt['sources']
        ):
            eval_rows.append({
                'sentence_id' : sid,
                'word'        : word,
                'category'    : cat,
                'is_oov'      : is_oov,
                'sentence'    : utt['sentence'],
                'source'      : src,
            })
        sentence_id += 1

eval_df = pd.DataFrame(eval_rows)
out_path = OUTPUT_DIR / 'evaluation_sentences.csv'
eval_df.to_csv(out_path, index=False, encoding='utf-8')

print(f'Saved → {out_path}')
print(f'Rows  : {len(eval_df)}')
print()
print(eval_df.head(10).to_string(index=False))

Saved → step5_results/evaluation_sentences.csv
Rows  : 1115

sentence_id            word category  is_oov                                                 sentence     source
   OOV_0001          भैरहवा      NAV    True  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस् seed_guide
   OOV_0001   discount code       CM    True  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस् seed_guide
   OOV_0001      दिल्लीबजार      NAV    True  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस् seed_guide
   OOV_0001           सिरहा     PROP    True  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस् seed_guide
   OOV_0001   login गर्नुस्       CM    True  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस् seed_guide
   OOV_0002           Ncell     CMPY    True Ncell, Saurya Airlines, online class, Samsung, Carlsberg seed_guide
   OOV_0002 Saurya Airlines     CMPY    True Ncell, Saurya Airlines, online class, Samsung, Carlsberg seed_guide
   OOV_0002    online class       C

In [ ]:
script_path = OUTPUT_DIR / 'recording_script.txt'

with open(script_path, 'w', encoding='utf-8') as f:
    f.write('# Nepali OOV TTS Benchmark — Recording Script\n')
    f.write(f'# Total utterances: {len(oov_utterances) + len(iv_utterances)}\n')
    f.write(f'# Words per utterance: {GROUP_SIZE}\n')
    f.write('#\n')
    f.write('# Instructions:\n')
    f.write('#   - Read each line clearly, pausing briefly between words.\n')
    f.write('#   - Each comma = a short pause.\n')
    f.write('#   - Do not add sentence intonation — treat each word independently.\n')
    f.write('#   - If unsure of a pronunciation, ask the language expert.\n')
    f.write('#\n')
    f.write('# -------------------------------------------------\n')
    f.write(f'# SECTION A — OOV Utterances ({len(oov_utterances)} utterances)\n')
    f.write('# -------------------------------------------------\n')
    for i, utt in enumerate(oov_utterances, 1):
        f.write(f'{utt["sentence"]}\n')

    f.write('\n')
    f.write('# -------------------------------------------------\n')
    f.write(f'# SECTION B — IV Utterances ({len(iv_utterances)} utterances)\n')
    f.write('# -------------------------------------------------\n')
    for i, utt in enumerate(iv_utterances, 1):
        f.write(f'{utt["sentence"]}\n')

print(f'Saved → {script_path}')

# Preview first 5 OOV and 5 IV lines
print('\n OOV sample (first 5)')
for utt in oov_utterances[:5]:
    print(' ', utt['sentence'])
print('\n IV sample (first 5)')
for utt in iv_utterances[:5]:
    print(' ', utt['sentence'])

Saved → step5_results/recording_script.txt

── OOV sample (first 5) ──
  भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस्
  Ncell, Saurya Airlines, online class, Samsung, Carlsberg
  तुम्बहल, सिनामङ्गल, FonePay, उज्ज्वला योजना, zoom meeting
  spam message, Disprin, साँखु, तिलिचो ताल, साँझतिर
  स्वर्णलक्ष्मी, भक्तपुर दरबार, स्थानीय सरकार, Vivo, रारा ताल

── IV sample (first 5) ──
  इडिभी, बैंकरहरू, कन्जुसाई, बताउनुहुन्छ, अग्रवालका
  डिउटी, नेपालीबाट, श्रमजीवी, योजनासहित, फोटोमार्फत
  दायरालाई, नेपालीहरुले, पटानीले, प्रोग्रामर, डाईभर्सन
  बर्षातको, कानूनतः, शुक्लागण्डकी, प्रोडुसरको, छाडौैं
  नेपोलिक, नेपालगंजमा, मिलनचोकका, जोनीको, टालटुले


In [43]:
annotated_rows = []

for utt_list, section in [(oov_utterances, 'OOV'), (iv_utterances, 'IV')]:
    for i, utt in enumerate(utt_list, 1):
        annotated_rows.append({
            'utterance_id'  : f'{section}_{i:04d}',
            'section'       : section,
            'sentence'      : utt['sentence'],
            'categories'    : '|'.join(utt['categories']),
            'is_oov_flags'  : '|'.join(str(v) for v in utt['is_oov_flags']),
            'word_count'    : len(utt['words']),
        })

ann_df = pd.DataFrame(annotated_rows)
ann_path = OUTPUT_DIR / 'recording_script_annotated.csv'
ann_df.to_csv(ann_path, index=False, encoding='utf-8')

print(f'Saved → {ann_path}')
print(f'Rows  : {len(ann_df)}')
print()
print(ann_df.head(5).to_string(index=False))

Saved → step5_results/recording_script_annotated.csv
Rows  : 223

utterance_id section                                                    sentence               categories             is_oov_flags  word_count
    OOV_0001     OOV     भैरहवा, discount code, दिल्लीबजार, सिरहा, login गर्नुस्       NAV|CM|NAV|PROP|CM True|True|True|True|True           5
    OOV_0002     OOV    Ncell, Saurya Airlines, online class, Samsung, Carlsberg CMPY|CMPY|CM|BRAND|BRAND True|True|True|True|True           5
    OOV_0003     OOV   तुम्बहल, सिनामङ्गल, FonePay, उज्ज्वला योजना, zoom meeting   PROP|NAV|BRAND|GOVT|CM True|True|True|True|True           5
    OOV_0004     OOV           spam message, Disprin, साँखु, तिलिचो ताल, साँझतिर   CM|BRAND|NAV|PROP|PROP True|True|True|True|True           5
    OOV_0005     OOV स्वर्णलक्ष्मी, भक्तपुर दरबार, स्थानीय सरकार, Vivo, रारा ताल PROP|NAV|GOVT|BRAND|PROP True|True|True|True|True           5


In [44]:
print('=' * 60)
print(f'{"Category":8s}  {"OOV words":>10s}  {"IV words":>9s}  {"OOV utts":>9s}  {"IV utts":>8s}')
print('-' * 60)

total_oov_w = total_iv_w = 0

for cat in CATEGORIES:
    cat_oov = [r for r in oov_records if r['category'] == cat]
    cat_iv  = [r for r in iv_records  if r['category'] == cat]

    # Count how many utterances contain at least one word from this category
    oov_utts = sum(1 for u in oov_utterances if cat in u['categories'])
    iv_utts  = sum(1 for u in iv_utterances  if cat in u['categories'])

    print(f'  {cat:6s}    {len(cat_oov):>10d}  {len(cat_iv):>9d}  {oov_utts:>9d}  {iv_utts:>8d}')
    total_oov_w += len(cat_oov)
    total_iv_w  += len(cat_iv)

print('-' * 60)
print(f'  {"TOTAL":6s}    {total_oov_w:>10d}  {total_iv_w:>9d}  '
      f'{len(oov_utterances):>9d}  {len(iv_utterances):>8d}')
print('=' * 60)
print(f'\nGrand total utterances : {len(oov_utterances) + len(iv_utterances)}')
print(f'Estimated recording time (@ 10 sec/utt) : '
      f'{(len(oov_utterances)+len(iv_utterances))*10/60:.0f} min')

Category   OOV words   IV words   OOV utts   IV utts
------------------------------------------------------------
  ABBR              75         74         60        59
  BRAND             75         75         60        55
  CM                75         75         59        52
  CMPY              75         74         60        52
  GOVT              75         75         57        56
  PROP             150         74         91        57
  NAV               74         75         62        55
------------------------------------------------------------
  TOTAL            599        522        119       104

Grand total utterances : 223
Estimated recording time (@ 10 sec/utt) : 37 min


In [45]:
print('Running quality checks...\n')
passed = True

# Check 1: No word appears in more than one utterance
all_utt_words = []
for utt in oov_utterances + iv_utterances:
    all_utt_words.extend(utt['words'])

word_counts = pd.Series(all_utt_words).value_counts()
dupes = word_counts[word_counts > 1]
if len(dupes) == 0:
    print('✓  No duplicate words across utterances')
else:
    print(f'✗  {len(dupes)} words appear in multiple utterances:')
    print(dupes.head(10))
    passed = False

# Check 2: All utterances have exactly GROUP_SIZE words
wrong_size = [u for u in oov_utterances + iv_utterances if len(u['words']) != GROUP_SIZE]
if len(wrong_size) == 0:
    print(f'✓  All utterances have exactly {GROUP_SIZE} words')
else:
    print(f'✗  {len(wrong_size)} utterances have wrong size')
    passed = False

# Check 3: Each category has >= 50 OOV and >= 50 IV words in eval set
print()
min_required = 50
for cat in CATEGORIES:
    n_oov = len([r for r in oov_records if r['category'] == cat])
    n_iv  = len([r for r in iv_records  if r['category'] == cat])
    oov_ok = '✓' if n_oov >= min_required else '✗'
    iv_ok  = '✓' if n_iv  >= min_required else '✗'
    print(f'  {cat:6s}  OOV={n_oov:3d} {oov_ok}   IV={n_iv:3d} {iv_ok}')
    if n_oov < min_required or n_iv < min_required:
        passed = False

print()
if passed:
    print('✅ All quality checks passed. Ready for recording.')
else:
    print('⚠️  Some checks failed — review above before proceeding.')

Running quality checks...

✓  No duplicate words across utterances
✓  All utterances have exactly 5 words

  ABBR    OOV= 75 ✓   IV= 74 ✓
  BRAND   OOV= 75 ✓   IV= 75 ✓
  CM      OOV= 75 ✓   IV= 75 ✓
  CMPY    OOV= 75 ✓   IV= 74 ✓
  GOVT    OOV= 75 ✓   IV= 75 ✓
  PROP    OOV=150 ✓   IV= 74 ✓
  NAV     OOV= 74 ✓   IV= 75 ✓

✅ All quality checks passed. Ready for recording.


In [46]:
per_cat = {}
for cat in CATEGORIES:
    per_cat[cat] = {
        'n_oov_words' : len([r for r in oov_records if r['category'] == cat]),
        'n_iv_words'  : len([r for r in iv_records  if r['category'] == cat]),
    }

summary = {
    'description'           : 'Step 5 — Evaluation sentences for Nepali OOV TTS benchmark',
    'group_size'            : GROUP_SIZE,
    'prop_oov_cap'          : PROP_OOV_CAP,
    'random_seed'           : SEED,
    'n_oov_words_total'     : len(oov_records),
    'n_iv_words_total'      : len(iv_records),
    'n_oov_utterances'      : len(oov_utterances),
    'n_iv_utterances'       : len(iv_utterances),
    'n_total_utterances'    : len(oov_utterances) + len(iv_utterances),
    'est_recording_min'     : round((len(oov_utterances)+len(iv_utterances))*10/60, 1),
    'per_category'          : per_cat,
    'output_files'          : [
        'evaluation_sentences.csv',
        'recording_script.txt',
        'recording_script_annotated.csv',
        'summary.json',
    ]
}

summary_path = OUTPUT_DIR / 'summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'Saved → {summary_path}')
print()
print(json.dumps(summary, ensure_ascii=False, indent=2))

Saved → step5_results/summary.json

{
  "description": "Step 5 — Evaluation sentences for Nepali OOV TTS benchmark",
  "group_size": 5,
  "prop_oov_cap": 150,
  "random_seed": 42,
  "n_oov_words_total": 599,
  "n_iv_words_total": 522,
  "n_oov_utterances": 119,
  "n_iv_utterances": 104,
  "n_total_utterances": 223,
  "est_recording_min": 37.2,
  "per_category": {
    "ABBR": {
      "n_oov_words": 75,
      "n_iv_words": 74
    },
    "BRAND": {
      "n_oov_words": 75,
      "n_iv_words": 75
    },
    "CM": {
      "n_oov_words": 75,
      "n_iv_words": 75
    },
    "CMPY": {
      "n_oov_words": 75,
      "n_iv_words": 74
    },
    "GOVT": {
      "n_oov_words": 75,
      "n_iv_words": 75
    },
    "PROP": {
      "n_oov_words": 150,
      "n_iv_words": 74
    },
    "NAV": {
      "n_oov_words": 74,
      "n_iv_words": 75
    }
  },
  "output_files": [
    "evaluation_sentences.csv",
    "recording_script.txt",
    "recording_script_annotated.csv",
    "summary.json"
  ]
}


In [47]:
print('=' * 58)
print('  STEP 5 COMPLETE — READINESS REPORT')
print('=' * 58)
print(f'  evaluation_sentences.csv      : {len(eval_df):5d} rows')
print(f'  recording_script.txt          : {len(oov_utterances)+len(iv_utterances):5d} utterances')
print(f'  recording_script_annotated.csv: {len(ann_df):5d} rows')
print()
print(f'  OOV utterances : {len(oov_utterances)}')
print(f'  IV  utterances : {len(iv_utterances)}')
print(f'  Estimated recording time : ~{round((len(oov_utterances)+len(iv_utterances))*10/60)} min per speaker')
print()
print('  Next step → Step 6: Record with Volunteers')
print('  Files: step5_results/')
print('=' * 58)

  STEP 5 COMPLETE — READINESS REPORT
  evaluation_sentences.csv      :  1115 rows
  recording_script.txt          :   223 utterances
  recording_script_annotated.csv:   223 rows

  OOV utterances : 119
  IV  utterances : 104
  Estimated recording time : ~37 min per speaker

  Next step → Step 6: Record with Volunteers
  Files: step5_results/
